In [1]:
# Setup — install project dependencies.
# Clones the semantic-correspondence repository if needed and installs the Python packages required by the notebook.
# Run once at the start of a fresh Colab session.

!test -d /content/semantic-correspondence || git clone -q https://github.com/aexomir/semantic-correspondence.git /content/semantic-correspondence
!pip -q install -U pip
!pip -q install -r /content/semantic-correspondence/requirements.txt
!pip -q install opencv-python tqdm
!pip -q install git+https://github.com/facebookresearch/segment-anything.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# Setup — clone local DINO repositories.
# DINOv2 and DINOv3 are loaded from local GitHub checkouts because they are not installed as normal PyPI packages here.

!rm -rf dinov2 dinov3
!git clone -q https://github.com/facebookresearch/dinov2.git
!git clone -q https://github.com/facebookresearch/dinov3.git

!pip -q install omegaconf torchmetrics fvcore iopath submitit ftfy regex scikit-learn termcolor

In [3]:
# Setup — imports, Google Drive mount, and global paths.
# Defines model weight paths, dataset locations, and the local Colab workspace used by later cells.

import os
import sys
import shutil
import types
import math
import json
import cv2
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from contextlib import nullcontext
import pandas as pd
from google.colab import drive

# Connect to Google Drive (will ask for permission the first time)
drive.mount('/content/drive', force_remount=True)

MYDRIVE = '/content/drive/MyDrive'

# Paths to saved model weights on Drive
DINOV2_WEIGHTS = os.path.join(MYDRIVE, 'dinov2_vitl14_pretrain_big.pth')
DINOV3_WEIGHTS = os.path.join(MYDRIVE, 'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth')
SAM_WEIGHTS_B  = os.path.join(MYDRIVE, 'sam_vit_b_01ec64_small.pth')

# Paths to dataset files on Drive
SPAIR_DIR = os.path.join(MYDRIVE, 'SPair-71k')
SPAIR_TAR = os.path.join(MYDRIVE, 'SPair-71k.tar.gz')
PFW_ZIP   = os.path.join(MYDRIVE, 'pf-willow.zip')

# Local fast disk folder (on Colab's machine, not Drive)
LOCAL_DATA_DIR = '/content/data'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

if not os.path.isdir(MYDRIVE):
    raise RuntimeError('Drive not mounted. Complete the auth prompt and re-run this cell.')

print('OK:', MYDRIVE)


Mounted at /content/drive
OK: /content/drive/MyDrive


In [4]:
# Setup — prepare the SPair-71k dataset locally.
# Uses the tar archive when available; otherwise copies the existing Drive folder into Colab local storage.

local_spair = os.path.join(LOCAL_DATA_DIR, 'SPair-71k')

if not os.path.exists(local_spair):
    if os.path.exists(SPAIR_TAR):
        print(f"Extracting {SPAIR_TAR} to {LOCAL_DATA_DIR}...")
        !tar -xzf {SPAIR_TAR} -C {LOCAL_DATA_DIR}
        print("Extraction complete!")
    elif os.path.exists(SPAIR_DIR):
        print(f"tar file not found. Copying directory {SPAIR_DIR} to {local_spair}...")
        shutil.copytree(SPAIR_DIR, local_spair)
        print("Copy complete!")
    else:
        print("Error: Neither tar file nor directory found in Drive.")
else:
    print(f"Dataset already exists at {local_spair}")


In [5]:
# Setup — load pretrained backbone models.
# Loads DINOv2, DINOv3, and SAM, moves them to the selected device, and verifies GPU placement.

from segment_anything import SamPredictor, sam_model_registry

# Load SAM ViT-B
sam = sam_model_registry['vit_b'](checkpoint=SAM_WEIGHTS_B)
sam_predictor = SamPredictor(sam)

# Load DINOv2 ViT-L/14.
dinov2 = torch.hub.load('dinov2', 'dinov2_vitl14', source='local', weights=DINOV2_WEIGHTS)

# Load DINOv3 ViT-B/16 with the matching ViT-B checkpoint.
dinov3 = torch.hub.load('dinov3', 'dinov3_vitb16', source='local', weights=DINOV3_WEIGHTS)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
sam.to(device).eval()
dinov2.to(device).eval()
dinov3.to(device).eval()

print('Ready. Device:', device)

# ── GPU sanity checks ────────────────────────────────────────────────────
print('DINOv2 device:', next(dinov2.parameters()).device)
print('DINOv3 device:', next(dinov3.parameters()).device)
print('SAM    device:', next(sam.parameters()).device)
assert str(next(dinov2.parameters()).device).startswith('cuda'), \
    'WARNING: DINOv2 is NOT on GPU! Check runtime type in Runtime > Change runtime type.'


/content/dinov2/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/content/dinov2/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/content/dinov2/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Ready. Device: cuda
DINOv2 device: cuda:0
DINOv3 device: cuda:0
SAM    device: cuda:0


In [6]:
# Setup — feature extraction helpers.
# Converts each model's output into a spatial feature map with shape (1, D, H_feat, W_feat).

def extract_dino_features(model, img_tensor, trainable=False):
    """
    Run an image through a DINO model and return its patch features.

    The model splits the image into a grid of small patches and produces
    one feature vector per patch. We reshape these vectors into a 2D
    spatial grid so they are easy to work with.

    Input:  img_tensor — a torch tensor of shape (1, C, H, W)
            trainable  — if True, keeps gradients (for finetuning);
                         if False, runs under no_grad and forces eval mode
    Output: torch tensor of shape (1, D, grid_H, grid_W)
              D = feature dimension, grid_H/W = number of patches per side
    """
    if not trainable:
        model.eval()

    ctx = torch.no_grad() if not trainable else nullcontext()
    with ctx:
        out = model.forward_features(img_tensor)
        # Different DINO versions store patch tokens under different key names
        if isinstance(out, dict):
            patch_tokens = out.get('x_norm_patchtokens', out.get('x_norm_patch_tokens'))
        else:
            patch_tokens = out

        if patch_tokens is None:
            keys = list(out.keys()) if isinstance(out, dict) else 'N/A'
            raise ValueError(f'Could not find patch tokens. Keys: {keys}')

        b, n, d = patch_tokens.shape
        grid = int(math.isqrt(n))  # num_patches must be a perfect square (e.g. 37*37=1369)
        if grid * grid != n:
            raise ValueError(f'Expected square number of patches, got N={n}')

        # Rearrange from (batch, num_patches, dim) -> (batch, dim, grid_H, grid_W)
        return patch_tokens.permute(0, 2, 1).reshape(b, d, grid, grid)


def extract_dino_layers(model, img_tensor, layer_ids):
    """
    Run an image through DINO and return features from specific middle layers.

    Instead of just the final output, we can inspect features at any transformer
    block. This is useful when fine-tuning — you want to unfreeze only the last
    few layers and see which layers help the most.

    Input:  img_tensor — torch tensor (1, C, H, W)
            layer_ids  — list of layer indices to extract, e.g. [10, 11]
    Output: dict mapping each layer_id -> torch tensor (1, D, grid_H, grid_W)
    """
    model.eval()
    with torch.no_grad():
        # Get the output of every transformer block at once
        all_layers = model.get_intermediate_layers(img_tensor, n=len(model.blocks), norm=True)
        out = {}
        for lid in layer_ids:
            patch_tokens = all_layers[lid]
            b, n, d = patch_tokens.shape
            grid = int(math.isqrt(n))
            if grid * grid != n:
                raise ValueError(f'Layer {lid}: expected square patches, got N={n}')
            # Rearrange to (batch, dim, grid_H, grid_W)
            out[lid] = patch_tokens.permute(0, 2, 1).reshape(b, d, grid, grid)
        return out


def extract_sam_features(predictor, image_pil, res=1024):
    """Run an image through SAM's image encoder and return its feature map.

    To match SAM's pretrained setup (and avoid any positional-embedding
    interpolation), we run SAM at its native resolution: 1024×1024.

    Input:  image_pil — RGB PIL Image
            res       — must be 1024
    Output: torch tensor of shape (1, C, feat_H, feat_W)
    """
    if int(res) != 1024:
        raise ValueError(f"SAM without pos-embed resizing requires res=1024, got {res}")

    device = next(predictor.model.parameters()).device
    encoder = predictor.model.image_encoder

    # Resize image to 1024×1024
    image_resized = transforms.functional.resize(
        image_pil, (1024, 1024), interpolation=transforms.InterpolationMode.BILINEAR
    )

    # Convert to torch tensor (values in [0, 255]) and normalize using SAM's mean/std
    img_tensor = transforms.ToTensor()(image_resized).unsqueeze(0).to(device) * 255.0
    pixel_mean = torch.tensor([123.675, 116.28, 103.53], device=device).view(-1, 1, 1)
    pixel_std  = torch.tensor([58.395, 57.12, 57.375],  device=device).view(-1, 1, 1)
    img_tensor = (img_tensor - pixel_mean) / pixel_std

    with torch.no_grad():
        feats = encoder(img_tensor)

    return feats


In [7]:
# Setup — dataset paths, model registry, and feature cache helpers.
# Keeps cached features on local Colab storage for speed and provides utilities to sync them back to Drive.

SPAIR_ROOT = os.path.join(LOCAL_DATA_DIR, 'SPair-71k')
JPEG_ROOT = os.path.join(SPAIR_ROOT, 'JPEGImages')              # folder with all the images
PAIRANN_TEST_ROOT = os.path.join(SPAIR_ROOT, 'PairAnnotation', 'test')  # folder with keypoint labels

# Features are cached on local SSD; sync to Drive once per session.
LOCAL_FEATURE_ROOT = '/content/features/spair71k'   # fast local SSD cache
DRIVE_FEATURE_ROOT = os.path.join(MYDRIVE, 'features', 'spair71k')  # persistent Drive copy
FEATURE_ROOT = LOCAL_FEATURE_ROOT   # all cache reads/writes go here
RESULTS_ROOT = os.path.join(MYDRIVE, 'results')                 # where we save evaluation results

# --- Model registry ---
# Each entry says: which model object to use, what image size it needs, and what type it is.
# 'dino' models use the torchvision preprocessing pipeline.
# 'sam' models use SAM's own preprocessing inside extract_sam_features.
MODEL_SPECS = {
    'dinov2_vitl14': {
        'model': dinov2,
        'img_size': (518, 518),   # 518 / 14 approx 37 patches per side
        'kind': 'dino',
    },
    'dinov3_vitb16': {
        'model': dinov3,
        'img_size': (512, 512),   # 512 / 16 = 32 patches per side
        'kind': 'dino',
    },
    'sam_vit_b_res1024': {
        'model': sam_predictor,
        'img_size': (1024, 1024),
        'kind': 'sam',
        'sam_res': 1024,
    },
}


def _ensure_parent_dir(path):
    """Create the folder that contains the given file path, if it does not exist yet."""
    os.makedirs(os.path.dirname(path), exist_ok=True)


def _rel_from_jpeg_root(image_path):
    """Return the path of an image relative to the dataset images folder (JPEG_ROOT)."""
    rel = os.path.relpath(image_path, JPEG_ROOT)
    if rel.startswith('..'):
        raise ValueError(f'Expected image under JPEG_ROOT={JPEG_ROOT}, got {image_path}')
    return rel


def feature_path(model_key, image_path, layer_id=None):
    """Build the file path where we store cached features for one image.

    The path mirrors the dataset folder structure so features are easy to find.
    Example: features/spair71k/dinov2_vitl14/dog/image001.pt
    """
    rel = _rel_from_jpeg_root(image_path)
    rel_no_ext = os.path.splitext(rel)[0]
    # If layer_id is provided (DINO intermediate layer), cache it separately
    layer_tag = '' if layer_id is None else f'_layer{int(layer_id)}'
    return os.path.join(FEATURE_ROOT, model_key, rel_no_ext + layer_tag + '.pt')


def save_feature_pt(path, feat_tensor):
    """Save a feature tensor to a .pt file. Creates the parent folder if needed."""
    _ensure_parent_dir(path)
    torch.save(feat_tensor, path)


def load_feature_pt(path):
    """Load and return the feature tensor stored in a .pt file."""
    return torch.load(path, map_location='cpu')


def dino_transform(img_size):
    """
    Build a torchvision preprocessing pipeline for DINO models.

    Three steps:
      1. Resize the image to img_size using bicubic interpolation
      2. Convert pixel values from [0, 255] integers to [0.0, 1.0] floats (ToTensor)
      3. Normalize with ImageNet mean and std (what DINO was trained with)

    Returns a torchvision Compose object — call it like a function on a PIL image.
    """
    return transforms.Compose(
        [
            transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )


# Pre-build one transform pipeline per DINO model so we don't recreate it each time
_DINO_TRANSFORMS = {
    k: dino_transform(v['img_size'])
    for k, v in MODEL_SPECS.items()
    if v['kind'] == 'dino'
}


def compute_feature(model_key, image_path, layer_id=None):
    """
    Load one image, run it through the specified model, and return its feature map
    as a float16 torch tensor stored on CPU.

    For DINO models:
      - Apply the torchvision preprocessing pipeline (_DINO_TRANSFORMS)
      - Add a batch dimension and move to GPU
      - Run extract_dino_features -> get a torch tensor (1, D, H, W)

    For SAM:
      - Open image with PIL, convert to raw array for cv2
      - Run extract_sam_features -> get a torch tensor (1, C, H, W)

    Returns a float16 CPU tensor (D, H, W).
    Storing as float16 halves the disk space compared to float32.
    """
    spec = MODEL_SPECS[model_key]
    kind = spec['kind']

    img_pil = Image.open(image_path).convert('RGB')

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    if kind == 'dino':
        t = _DINO_TRANSFORMS[model_key](img_pil).unsqueeze(0).to(device)  # (1, C, H, W)
        if layer_id is None:
            feat = extract_dino_features(spec['model'], t)                     # (1, D, H, W)
        else:
            feat = extract_dino_layers(spec['model'], t, [int(layer_id)])[int(layer_id)]  # (1, D, H, W)
    elif kind == 'sam':
        feat = extract_sam_features(spec['model'], img_pil, res=spec['sam_res'])  # (1, C, H, W)
    else:
        raise ValueError(f'Unknown kind: {kind}')

    # Remove batch dim, store as float16 on CPU to save memory
    return feat.squeeze(0).detach().half().cpu()


def load_or_compute_feature(model_key, image_path, layer_id=None):
    """
    Return the feature tensor for one image, using the disk cache when possible.

    - If a cached .pt file already exists on Drive -> load and return it immediately (fast)
    - If not -> run the model, save the result to Drive, then return it

    This means each image is processed by the model only once across all runs.
    """
    out_path = feature_path(model_key, image_path, layer_id=layer_id)
    if os.path.exists(out_path):
        return load_feature_pt(out_path)

    feat = compute_feature(model_key, image_path, layer_id=layer_id)
    save_feature_pt(out_path, feat)
    return feat


os.makedirs(FEATURE_ROOT, exist_ok=True)
os.makedirs(RESULTS_ROOT, exist_ok=True)


def sync_features_to_drive():
    """
    Copy the local feature cache to Google Drive for persistence.
    Call this once after all features are computed / after fine-tuning.
    It only copies files that are missing on Drive, so it is safe to re-run.
    """
    print(f'Syncing {LOCAL_FEATURE_ROOT} -> {DRIVE_FEATURE_ROOT} ...')
    os.makedirs(DRIVE_FEATURE_ROOT, exist_ok=True)
    for root, dirs, files in os.walk(LOCAL_FEATURE_ROOT):
        rel = os.path.relpath(root, LOCAL_FEATURE_ROOT)
        dst_root = os.path.join(DRIVE_FEATURE_ROOT, rel)
        os.makedirs(dst_root, exist_ok=True)
        for fn in files:
            src_f = os.path.join(root, fn)
            dst_f = os.path.join(dst_root, fn)
            if not os.path.exists(dst_f):
                shutil.copy2(src_f, dst_f)
    print('Sync complete.')


def seed_local_cache_from_drive():
    """
    Copy any previously saved feature files from Drive to local SSD at startup.
    Run this once after mounting Drive so subsequent runs skip recomputation.
    """
    if not os.path.isdir(DRIVE_FEATURE_ROOT):
        print('No Drive cache found — starting fresh.')
        return
    print(f'Seeding local cache from {DRIVE_FEATURE_ROOT} ...')
    count = 0
    for root, dirs, files in os.walk(DRIVE_FEATURE_ROOT):
        rel = os.path.relpath(root, DRIVE_FEATURE_ROOT)
        dst_root = os.path.join(LOCAL_FEATURE_ROOT, rel)
        os.makedirs(dst_root, exist_ok=True)
        for fn in files:
            src_f = os.path.join(root, fn)
            dst_f = os.path.join(dst_root, fn)
            if not os.path.exists(dst_f):
                shutil.copy2(src_f, dst_f)
                count += 1
    print(f'Seeded {count} files into local cache.')


# Seed local SSD from Drive right away (fast copy, happens once per session)
seed_local_cache_from_drive()

print('Local cache root:', FEATURE_ROOT)
print('Drive  cache root:', DRIVE_FEATURE_ROOT)
print('Results root:', RESULTS_ROOT)
print('JPEG root:', JPEG_ROOT)
print('PairAnnotation test root:', PAIRANN_TEST_ROOT)


Local cache root: /content/features/spair71k
Drive  cache root: /content/drive/MyDrive/features/spair71k
Results root: /content/drive/MyDrive/results
JPEG root: /content/data/SPair-71k/JPEGImages
PairAnnotation test root: /content/data/SPair-71k/PairAnnotation/test


In [8]:
# Stage 1 — precompute and cache image features.
# Runs each selected backbone once over the dataset and stores feature tensors for later evaluation.

VALID_IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}


def iter_jpeg_images():
    """
    Walk through the dataset images folder and yield the full path
    of every valid image file (.jpg, .jpeg, .png).
    """
    for root, _, files in os.walk(JPEG_ROOT):
        for fn in files:
            ext = os.path.splitext(fn)[1].lower()
            if ext in VALID_IMAGE_EXTS:
                yield os.path.join(root, fn)


def precompute_features(model_keys=None, limit=None):
    """
    Run all images through each model and save their feature tensors to Drive.

    - model_keys: which models to run (default: all models in MODEL_SPECS)
    - limit: optionally process only the first N images (useful for quick tests)

    For each image, if the feature file already exists on Drive we skip it.
    This makes the function safe to re-run — it only processes new images.
    """
    if model_keys is None:
        model_keys = list(MODEL_SPECS.keys())

    images = list(iter_jpeg_images())
    if limit is not None:
        images = images[: int(limit)]

    print(f'Found {len(images)} images under {JPEG_ROOT}')
    print('Models:', model_keys)

    for model_key in model_keys:
        missing = 0
        for img_path in tqdm(images, desc=f'{model_key}: caching'):
            out_path = feature_path(model_key, img_path)
            if os.path.exists(out_path):
                continue  # already cached, skip

            feat = compute_feature(model_key, img_path)
            save_feature_pt(out_path, feat)
            missing += 1

        print(f'{model_key}: newly cached {missing} feature files')


# Example usage:
# precompute_features(['dinov2_vitl14'], limit=50)    # quick test on 50 images
# precompute_features(list(MODEL_SPECS.keys()))       # full run on all models


In [9]:
# Stage 1 — SPair-71k evaluation pipeline.
# Computes PCK by matching source keypoints to target feature locations using either argmax or window soft-argmax.

PCK_THRESHOLDS = [0.05, 0.1, 0.15, 0.2]


def list_pair_files(split="test"):
    """
    Find all JSON annotation files for a given SPair-71k split.
    split: one of "test", "val", "trn"
    Each JSON file describes one image pair and contains the
    source/target keypoints and bounding box info.
    """
    split_dir = os.path.join(SPAIR_ROOT, 'PairAnnotation', split)
    if not os.path.isdir(split_dir):
        raise FileNotFoundError(f'Missing PairAnnotation {split} dir: {split_dir}')

    pair_files = []
    for root, _, files in os.walk(split_dir):
        for fn in files:
            if fn.endswith('.json'):
                pair_files.append(os.path.join(root, fn))

    pair_files.sort()
    return pair_files


# Keep the old name as a convenience alias so Stage 1 call sites still work
def list_test_pair_files():
    return list_pair_files(split="test")


def _parse_xy(p):
    """
    Safely read a keypoint coordinate from the annotation.
    Returns (x, y) as floats, or None if the point is missing,
    invalid, or has negative coordinates (means it was not labeled).
    """
    if p is None:
        return None
    if not isinstance(p, (list, tuple)):
        return None
    if len(p) < 2:
        return None
    x, y = float(p[0]), float(p[1])
    if x < 0 or y < 0:
        return None
    return x, y


def softargmax_2d(similarity_map, temperature=0.01, window_size=5):
    """
    Find the best-matching location in a similarity map with sub-pixel accuracy.

    Plain argmax snaps to the nearest grid cell. This function does better:
      1. Find the peak grid cell with argmax
      2. Cut out a small window (e.g. 5x5) around that peak
      3. Apply softmax to the window values to get a probability distribution
      4. Compute the weighted average position — this gives a fractional (x, y)

    The temperature parameter controls how 'sharp' the softmax is.
    A low temperature (e.g. 0.01) makes it almost the same as argmax.

    Args:
        similarity_map: 2D torch tensor (H, W) of similarity scores
        temperature:    softmax temperature (lower = sharper)
        window_size:    size of the window around the peak

    Returns (pred_x, pred_y) in feature-map coordinates (sub-pixel precision).
    """
    H, W = similarity_map.shape
    device = similarity_map.device

    # Step 1: find the peak cell
    flat_idx = similarity_map.argmax().item()
    peak_y = flat_idx // W
    peak_x = flat_idx % W

    # Step 2: define window around the peak (clamped to map boundaries)
    half = window_size // 2
    y_start = max(0, peak_y - half)
    y_end   = min(H, peak_y + half + 1)
    x_start = max(0, peak_x - half)
    x_end   = min(W, peak_x + half + 1)

    window = similarity_map[y_start:y_end, x_start:x_end]

    # Step 3: softmax — apply temperature scaling
    window_flat = window.reshape(-1)
    probs = F.softmax(window_flat / temperature, dim=0)

    # Step 4: build coordinate grids for the window
    y_coords = torch.arange(y_start, y_end, device=device).float()
    x_coords = torch.arange(x_start, x_end, device=device).float()
    grid_y, grid_x = torch.meshgrid(y_coords, x_coords, indexing='ij')

    # Step 5: weighted average position
    pred_x = (probs * grid_x.reshape(-1)).sum()
    pred_y = (probs * grid_y.reshape(-1)).sum()
    return pred_x, pred_y


def evaluate_cached(
    model_key,
    use_softargmax,
    layer_id=None,
    softargmax_temp=0.01,
    softargmax_window=5,
    thresholds=PCK_THRESHOLDS,
    max_pairs=None,
    save_csv=True,
    bypass_cache=False,
    split="test",
):
    """
    Main evaluation function. Measures how well a model does at
    semantic correspondence on the SPair-71k test set.

    For each image pair:
      - Load cached feature tensors for source and target images
      - L2-normalize them with F.normalize
      - For each source keypoint, extract its feature vector
      - Compute cosine similarity with every patch in the target
      - Predict the match using argmax or soft-argmax
      - Check if the prediction is within threshold x bbox_size of the truth

    Parameters:
      model_key       — which model to evaluate (must be in MODEL_SPECS)
      use_softargmax  — if True, use window soft-argmax; if False, use plain argmax
      max_pairs       — limit to first N pairs (useful for quick tests)
      save_csv        — if True, save per-image results to a CSV on Drive
      split           — which SPair-71k split to evaluate on ("test", "val", "trn")

    Returns a dict with overall PCK scores and per-image results.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    pair_files = list_pair_files(split=split)
    if max_pairs is not None:
        pair_files = pair_files[: int(max_pairs)]

    correct_kps = {t: 0 for t in thresholds}  # count of correct predictions per threshold
    total_kps = 0                              # total keypoints seen

    per_image_results = []

    for pair_path in tqdm(pair_files, desc=f'Evaluating {model_key}'):
        # Load the annotation for this pair
        with open(pair_path, 'r') as f:
            ann = json.load(f)

        category = ann['category']
        src_name = ann['src_imname']
        trg_name = ann['trg_imname']

        src_img_path = os.path.join(JPEG_ROOT, category, src_name)
        trg_img_path = os.path.join(JPEG_ROOT, category, trg_name)

        # Get the pixel size of each image (needed to map feature coords back to pixels)
        src_pil = Image.open(src_img_path).convert('RGB')
        trg_pil = Image.open(trg_img_path).convert('RGB')
        src_w, src_h = src_pil.size
        trg_w, trg_h = trg_pil.size

        # Load (or compute) feature tensors, upcast to float32, L2-normalize, move to device
        # bypass_cache=True skips disk I/O and re-runs the model directly (used during fine-tuning
        # evaluation so we don't need to wipe and recompute the entire cache between runs).
        if bypass_cache:
            raw_src = compute_feature(model_key, src_img_path, layer_id=layer_id)
            raw_trg = compute_feature(model_key, trg_img_path, layer_id=layer_id)
        else:
            raw_src = load_or_compute_feature(model_key, src_img_path, layer_id=layer_id)
            raw_trg = load_or_compute_feature(model_key, trg_img_path, layer_id=layer_id)
        f_src = F.normalize(raw_src.float(), dim=0).to(device)
        f_trg = F.normalize(raw_trg.float(), dim=0).to(device)

        fh, fw = f_src.shape[1], f_src.shape[2]   # feature grid height and width
        if f_src.shape[1:] != f_trg.shape[1:]:
            raise ValueError(f'Feature grids mismatch: src={f_src.shape[1:]} trg={f_trg.shape[1:]}')

        src_kps = ann['src_kps']
        trg_kps = ann['trg_kps']

        # PCK threshold is normalized by the largest side of the target bounding box
        trg_bbox = ann['trg_bndbox']
        bbox_w = float(trg_bbox[2] - trg_bbox[0])
        bbox_h = float(trg_bbox[3] - trg_bbox[1])
        norm_factor = max(bbox_w, bbox_h)

        image_correct = {t: 0 for t in thresholds}
        image_total_kps = 0

        kp_indices = range(len(src_kps)) if isinstance(src_kps, list) else src_kps.keys()
        for idx in kp_indices:
            p_src = _parse_xy(src_kps[idx])
            p_trg = _parse_xy(trg_kps[idx])
            if p_src is None or p_trg is None:
                continue  # skip unlabeled or invalid keypoints

            sx, sy = p_src   # source keypoint in pixel coordinates
            tx, ty = p_trg   # target ground-truth in pixel coordinates

            # Convert source keypoint from pixel coords to feature grid coords
            feat_x = int(min(sx / src_w * fw, fw - 1))
            feat_y = int(min(sy / src_h * fh, fh - 1))
            feat_x = max(feat_x, 0)
            feat_y = max(feat_y, 0)

            # Extract the feature vector at that grid cell
            target_feat = f_src[:, feat_y, feat_x]  # shape: (D,)

            # Compute cosine similarity between this vector and every target patch
            sim = torch.einsum('c,chw->hw', target_feat, f_trg)  # shape: (fh, fw)

            # Find the best-matching location in the target feature map
            if use_softargmax:
                pred_x_feat, pred_y_feat = softargmax_2d(
                    sim,
                    temperature=softargmax_temp,
                    window_size=softargmax_window,
                )
                pred_x_feat = pred_x_feat.item()
                pred_y_feat = pred_y_feat.item()
            else:
                # Simple argmax: pick the single highest-similarity cell
                flat = sim.argmax().item()
                pred_y_feat = float(flat // fw)
                pred_x_feat = float(flat % fw)

            # Convert predicted feature-grid location back to pixel coordinates
            pred_x = ((pred_x_feat + 0.5) / fw) * trg_w
            pred_y = ((pred_y_feat + 0.5) / fh) * trg_h

            # Euclidean distance between prediction and ground truth (in pixels)
            dist = math.sqrt((pred_x - tx) ** 2 + (pred_y - ty) ** 2)

            total_kps += 1
            image_total_kps += 1

            # A prediction is 'correct' if distance <= threshold × bbox_size
            for t in thresholds:
                if dist <= (t * norm_factor):
                    correct_kps[t] += 1
                    image_correct[t] += 1

        per_image_results.append(
            {
                'category': category,
                'src_image': src_name,
                'trg_image': trg_name,
                'total_kps': image_total_kps,
                **{
                    f'PCK@{t}': (image_correct[t] / image_total_kps * 100.0)
                    if image_total_kps > 0
                    else 0.0
                    for t in thresholds
                },
            }
        )

    # Aggregate results: overall PCK per threshold
    per_keypoint = {
        f'PCK@{t}': (correct_kps[t] / total_kps * 100.0) if total_kps > 0 else 0.0
        for t in thresholds
    }

    results = {
        'model_key': model_key,
        'use_softargmax': bool(use_softargmax),
        'softargmax_temp': float(softargmax_temp),
        'softargmax_window': int(softargmax_window),
        'per_keypoint': per_keypoint,
        'per_image': per_image_results,
        'total_keypoints': int(total_kps),
        'total_image_pairs': int(len(per_image_results)),
    }

    print('\nPer-keypoint PCK:')
    for t in thresholds:
        print(f'  PCK@{t}: {per_keypoint["PCK@" + str(t)]:.2f}%')

    if save_csv:
        os.makedirs(RESULTS_ROOT, exist_ok=True)
        suffix = 'softargmax' if use_softargmax else 'argmax'
        out_csv = os.path.join(RESULTS_ROOT, f'{model_key}-{suffix}_per_image_results.csv')
        pd.DataFrame(per_image_results).to_csv(out_csv, index=False)
        print('Saved per-image CSV:', out_csv)

        out_csv_kp = os.path.join(RESULTS_ROOT, f'{model_key}-{suffix}_per_keypoint_results.csv')
        pd.DataFrame([{
            'model_key': model_key,
            'use_softargmax': bool(use_softargmax),
            'total_keypoints': int(total_kps),
            'total_image_pairs': int(len(per_image_results)),
            **per_keypoint,
        }]).to_csv(out_csv_kp, index=False)
        print('Saved per-keypoint CSV:', out_csv_kp)

    return results


# Example usage:
# results = evaluate_cached('dinov2_vitl14', use_softargmax=False, max_pairs=200)
# results = evaluate_cached('dinov2_vitl14', use_softargmax=True, softargmax_temp=0.01, softargmax_window=7, max_pairs=200)


In [10]:
# Stage 1 — run baseline feature caching and evaluation.
# Processes one backbone at a time and writes the frozen-baseline results to Drive.

eval_summaries = []
for model_key in MODEL_SPECS.keys():
    # Step 1: cache features for this model only
    precompute_features([model_key])

    # Step 2: evaluate immediately after caching
    res_argmax = evaluate_cached(
        model_key,
        use_softargmax=False,
        save_csv=True,
    )
    eval_summaries.append(
        {
            "model": model_key,
            "mode": "argmax",
            **res_argmax["per_keypoint"],
            "total_keypoints": res_argmax["total_keypoints"],
            "total_image_pairs": res_argmax["total_image_pairs"],
        }
    )

summary_df = pd.DataFrame(eval_summaries)
os.makedirs(RESULTS_ROOT, exist_ok=True)
summary_path = os.path.join(RESULTS_ROOT, "summary_per_keypoint_pck.csv")
summary_df.to_csv(summary_path, index=False)
print("\nSaved summary:", summary_path)
display(summary_df)

# Persist the local feature cache to Drive for future sessions
sync_features_to_drive()


# Stage 2 — Fine-tuning the last backbone layers

This stage unfreezes the last **N** trainable blocks of each backbone and fine-tunes them with SPair-71k keypoint supervision using an InfoNCE-style correspondence loss.

The planned runs are **N ∈ {1, 2, 4}** for each of the three backbones. The **N = 0** frozen baseline is reused from Stage 1 rather than retrained.

The next cells define the run grid, execute fine-tuning per backbone, and accumulate validation/test results in `summary_finetune.csv`.


In [11]:
# Stage 2 — define fine-tuning runs.
# Creates the 3 × 3 grid of backbone/N combinations and skips runs whose checkpoint already exists.

FINETUNE_RUNS = []

_BASE = {
    "lr":               5e-6,
    "epochs":           3,
    "max_train_pairs":  None,   # use all available training pairs
    "temperature":      0.07,
    "grad_accum_steps": 4,
    "use_amp":          True,
    "checkpoint_dir":   os.path.join(MYDRIVE, "checkpoints", "finetuned"),
}

for _model_key in ["dinov2_vitl14", "dinov3_vitb16", "sam_vit_b_res1024"]:
    for _N in [1, 2, 4]:
        # Auto-resume: Check if this run's checkpoint already exists on Drive
        expected_ckpt = os.path.join(_BASE["checkpoint_dir"], f"{_model_key}_last{_N}.pth")
        if os.path.exists(expected_ckpt):
            print(f"Skipping {_model_key} N={_N} because checkpoint already exists.")
            continue

        FINETUNE_RUNS.append({"model_key": _model_key, "unfreeze_n": _N, **_BASE})

print(f"\nTotal runs remaining: {len(FINETUNE_RUNS)}")
for r in FINETUNE_RUNS:
    print(f"  {r['model_key']}  N={r['unfreeze_n']}")


In [12]:
# Stage 2 — fine-tuning setup and helper functions.
# Defines weight reloading, layer freezing, the SPair training dataset, batching, and the InfoNCE loss.

import random
import numpy as np
from torch.utils.data import DataLoader

# Stage-2 local checkpoint staging (faster than reloading from Drive each run)
LOCAL_WEIGHTS_DIR = "/content/weights"
os.makedirs(LOCAL_WEIGHTS_DIR, exist_ok=True)

_LOCAL_WEIGHT_SOURCES = {
    "dinov2_vitl14": DINOV2_WEIGHTS,
    "dinov3_vitb16": DINOV3_WEIGHTS,
    "sam_vit_b_res1024": SAM_WEIGHTS_B,
}

_LOCAL_WEIGHT_PATHS = {
    key: os.path.join(LOCAL_WEIGHTS_DIR, os.path.basename(src_path))
    for key, src_path in _LOCAL_WEIGHT_SOURCES.items()
}


def stage_weights_to_local():
    copied = 0
    for key, src_path in _LOCAL_WEIGHT_SOURCES.items():
        dst_path = _LOCAL_WEIGHT_PATHS[key]
        if not os.path.exists(dst_path):
            shutil.copy2(src_path, dst_path)
            copied += 1
    print(f"Staged {copied} weight file(s) into {LOCAL_WEIGHTS_DIR}")


stage_weights_to_local()


# ── 1. Reload pretrained weights ──────────────────────────────────────────────
def reload_pretrained(model_key):
    global sam, sam_predictor
    local_weights_path = _LOCAL_WEIGHT_PATHS[model_key]

    if model_key == "dinov2_vitl14":
        dinov2.load_state_dict(torch.load(local_weights_path, map_location="cpu"))
        dinov2.to(device)
    elif model_key == "dinov3_vitb16":
        dinov3.load_state_dict(torch.load(local_weights_path, map_location="cpu"))
        dinov3.to(device)
    elif model_key == "sam_vit_b_res1024":
        sam = sam_model_registry["vit_b"](checkpoint=local_weights_path)
        sam_predictor = SamPredictor(sam)
        sam.to(device)
        MODEL_SPECS["sam_vit_b_res1024"]["model"] = sam_predictor
    print(f"  Reloaded pretrained weights: {model_key}")


# ── 2. Freeze all / unfreeze last N blocks ────────────────────────────────────
def _get_backbone(model_key):
    if model_key == "dinov2_vitl14":
        return dinov2
    elif model_key == "dinov3_vitb16":
        return dinov3
    else:  # sam_vit_b_res1024
        return sam_predictor.model.image_encoder


def set_unfreeze(model_key, N):
    backbone = _get_backbone(model_key)
    for p in backbone.parameters():
        p.requires_grad = False
    if N > 0:
        for block in backbone.blocks[-N:]:
            for p in block.parameters():
                p.requires_grad = True
        for attr in ("norm", "neck"):
            if hasattr(backbone, attr):
                for p in getattr(backbone, attr).parameters():
                    p.requires_grad = True
    trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in backbone.parameters())
    print(f"  Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")


# ── 3. Training-mode feature extraction (gradients enabled) ───────────────────
def extract_ft(model_key, img_tensor):
    if model_key in ("dinov2_vitl14", "dinov3_vitb16"):
        model = dinov2 if model_key == "dinov2_vitl14" else dinov3
        out   = model.forward_features(img_tensor)
        if isinstance(out, dict):
            patch_tokens = out.get("x_norm_patchtokens", out.get("x_norm_patch_tokens"))
        else:
            patch_tokens = out
        b, n, d = patch_tokens.shape
        g = int(math.isqrt(n))
        return patch_tokens.permute(0, 2, 1).reshape(b, d, g, g)
    else:  # SAM
        return sam_predictor.model.image_encoder(img_tensor)


# ── 4. SPair training dataset ─────────────────────────────────────────────────
def _sam_preprocess(pil_img, res):
    arr  = cv2.resize(np.array(pil_img), (res, res)).astype(np.float32)
    t    = torch.from_numpy(arr).permute(2, 0, 1)
    mean = torch.tensor([123.675, 116.28, 103.53]).view(-1, 1, 1)
    std  = torch.tensor([58.395,  57.12,  57.375]).view(-1, 1, 1)
    return (t - mean) / std


class SPairDataset(torch.utils.data.Dataset):
    def __init__(self, root, model_key, max_pairs=None, split="trn"):  # None = use all pairs
        if split not in ("trn", "val", "test"):
            raise ValueError(
                f"SPairDataset: split must be one of 'trn', 'val', 'test' (got {split!r})"
            )
        self.image_dir = os.path.join(root, "JPEGImages")
        self.model_key = model_key
        self.split     = split
        pair_ann_root  = os.path.join(root, "PairAnnotation", split)
        if not os.path.isdir(pair_ann_root):
            raise FileNotFoundError(
                f"SPairDataset: missing annotation directory for split={split!r}: "
                f"{pair_ann_root}"
            )
        all_files = [
            os.path.join(r, fn)
            for r, _, fs in os.walk(pair_ann_root)
            for fn in fs if fn.endswith(".json")
        ]
        random.seed(42)
        random.shuffle(all_files)
        self.pair_files = all_files if max_pairs is None else all_files[:max_pairs]
        print(
            f"SPairDataset [{model_key}][split={split}]: "
            f"{len(self.pair_files)} / {len(all_files)} pairs "
            f"(all={max_pairs is None}) — {pair_ann_root}"
        )

    def __len__(self):
        return len(self.pair_files)

    def __getitem__(self, idx):
        with open(self.pair_files[idx]) as f:
            ann = json.load(f)
        cat     = ann["category"]
        src_pil = Image.open(os.path.join(self.image_dir, cat, ann["src_imname"])).convert("RGB")
        trg_pil = Image.open(os.path.join(self.image_dir, cat, ann["trg_imname"])).convert("RGB")
        sw, sh  = src_pil.size
        tw, th  = trg_pil.size

        spec = MODEL_SPECS[self.model_key]
        if spec["kind"] == "dino":
            t          = _DINO_TRANSFORMS[self.model_key]
            src_tensor = t(src_pil)
            trg_tensor = t(trg_pil)
        else:
            res        = spec["sam_res"]
            src_tensor = _sam_preprocess(src_pil, res)
            trg_tensor = _sam_preprocess(trg_pil, res)

        src_kps, trg_kps = ann["src_kps"], ann["trg_kps"]
        kp_indices = range(len(src_kps)) if isinstance(src_kps, list) else src_kps.keys()
        kps = []
        for i in kp_indices:
            ps, pt = src_kps[i], trg_kps[i]
            # SPair-71k uses negative coordinates (e.g. [-1, -1]) for unlabeled keypoints.
            # Reuse the same filtering rule as Stage 1 (_parse_xy rejects negatives).
            ps_xy = _parse_xy(ps)
            pt_xy = _parse_xy(pt)
            if ps_xy is None or pt_xy is None:
                continue
            kps.append({"src": (ps_xy[0] / sw, ps_xy[1] / sh),
                         "trg": (pt_xy[0] / tw, pt_xy[1] / th)})
        return {"src": src_tensor, "trg": trg_tensor, "kps": kps}


def _collate_fn(batch):
    return {
        "src": torch.stack([b["src"] for b in batch]),
        "trg": torch.stack([b["trg"] for b in batch]),
        "kps": [b["kps"] for b in batch],
    }


# ── 5. Contrastive correspondence loss (InfoNCE) ──────────────────────────────
def correspondence_loss(f_src, f_trg, kps_list, temperature=0.07):
    """
    f_src, f_trg : (B, D, H, W)
    kps_list     : list of B lists, each [{src: (x_n, y_n), trg: (x_n, y_n)}, ...]
    """
    B, D, H, W = f_src.shape
    device = f_src.device
    losses = []

    for b in range(B):
        trg_all = F.normalize(f_trg[b].view(D, -1), dim=0)

        src_descs = []
        targets = []
        for kp in kps_list[b]:
            sx = min(max(int(round(kp["src"][0] * (W - 1))), 0), W - 1)
            sy = min(max(int(round(kp["src"][1] * (H - 1))), 0), H - 1)
            tx = min(max(int(round(kp["trg"][0] * (W - 1))), 0), W - 1)
            ty = min(max(int(round(kp["trg"][1] * (H - 1))), 0), H - 1)
            src_descs.append(f_src[b, :, sy, sx])
            targets.append(ty * W + tx)

        if not src_descs:
            continue

        src_descs = F.normalize(torch.stack(src_descs, dim=0), dim=1)
        logits = torch.matmul(src_descs, trg_all) / temperature
        targets = torch.tensor(targets, dtype=torch.long, device=device)
        losses.append(F.cross_entropy(logits, targets, reduction="none"))

    return torch.cat(losses).mean() if losses else torch.tensor(0.0, device=device)


print("Stage 2 helpers defined.")

Staged 3 weight file(s) into /content/weights
Stage 2 helpers defined.


In [13]:
# Stage 2 — train and evaluate one backbone's fine-tuning runs.
# For each N, trains on the training split, selects checkpoints by validation PCK, and reports final test results.

def run_finetune_for_backbone(runs):
    """Train + evaluate one backbone's set of (N=1, 2, 4) runs.

    Designed to be called from a separate cell per backbone so each
    backbone's training + evaluation is self-contained and re-runnable.

    For each run we:
      1. Reload pretrained weights and unfreeze last N blocks
      2. Train with InfoNCE for cfg["epochs"] on the FULL
         PairAnnotation/trn split. After every epoch, evaluate on the
         val split (max_pairs=300) and save the checkpoint only when
         val PCK@0.1 improves (best-model checkpointing).
      3. After all training in this backbone is done, run a full-test
         evaluation on PairAnnotation/test for each N. This is the only
         place the test split is ever touched.

    Final full-test results are upserted into a shared CSV on Drive
    (drop existing rows for this backbone, then append fresh ones)
    so re-running this cell refreshes only this backbone's rows
    without disturbing other backbones already saved in the CSV.
    """
    if not runs:
        print("Nothing to run.")
        return

    os.makedirs(runs[0]["checkpoint_dir"], exist_ok=True)
    csv_path = os.path.join(RESULTS_ROOT, "summary_finetune.csv")

    full_rows = []
    checkpoint_paths = {}

    for cfg in runs:
        model_key = cfg["model_key"]
        N         = cfg["unfreeze_n"]
        print(f"\n{'='*60}")
        print(f"  {model_key}  |  unfreeze last N={N}")
        print(f"{'='*60}")

        reload_pretrained(model_key)
        set_unfreeze(model_key, N)

        # Drive path — the authoritative checkpoint location for resuming across sessions.
        ckpt = os.path.join(cfg["checkpoint_dir"], f"{model_key}_last{N}.pth")
        # Local SSD staging path — used during training to avoid per-epoch Drive writes.
        # Drive I/O is 10–50× slower than local SSD; we only copy here once, after all
        # epochs finish, so crash recovery still works (the local file is always current).
        local_ckpt = os.path.join(LOCAL_WEIGHTS_DIR, f"{model_key}_last{N}.pth")

        if N > 0:
            # Stage-2 fine-tuning trains on the FULL PairAnnotation/val split.
            # We never touch PairAnnotation/test during training — that split
            # is reserved for the final evaluation pass below.
            dataset  = SPairDataset(SPAIR_ROOT, model_key, max_pairs=None, split="trn")
            loader   = DataLoader(
                dataset,
                batch_size=1,
                shuffle=True,
                num_workers=2,
                pin_memory=True,
                persistent_workers=True,
                prefetch_factor=2,
                collate_fn=_collate_fn,
            )
            backbone = _get_backbone(model_key)
            opt      = torch.optim.AdamW(
                           filter(lambda p: p.requires_grad, backbone.parameters()),
                           lr=cfg["lr"])
            use_amp  = cfg["use_amp"] and device == "cuda"
            scaler   = torch.amp.GradScaler("cuda", enabled=use_amp)

            backbone.train()
            best_val_pck = 0.0
            for epoch in range(cfg["epochs"]):
                opt.zero_grad(set_to_none=True)
                running = 0.0
                for step, batch in enumerate(tqdm(loader, desc=f"ep{epoch+1}/{cfg['epochs']}")):
                    src = batch["src"].to(device)
                    trg = batch["trg"].to(device)
                    kps = batch["kps"]
                    with torch.autocast("cuda", enabled=use_amp):
                        merged_inputs = torch.cat([src, trg], dim=0)
                        merged_feats = extract_ft(model_key, merged_inputs)
                        src_feats, trg_feats = torch.split(merged_feats, src.shape[0], dim=0)
                        loss = correspondence_loss(
                            src_feats,
                            trg_feats,
                            kps,
                            cfg["temperature"],
                        ) / cfg["grad_accum_steps"]
                    scaler.scale(loss).backward()
                    running += loss.item() * cfg["grad_accum_steps"]
                    if (step + 1) % cfg["grad_accum_steps"] == 0:
                        scaler.step(opt)
                        scaler.update()
                        opt.zero_grad(set_to_none=True)
                print(f"  epoch {epoch + 1} avg loss: {running / len(loader):.4f}")

                # Per-epoch validation on the full val split (test split never touched here).
                backbone.eval()
                val_results = evaluate_cached(
                    model_key,
                    use_softargmax=False,
                    save_csv=False,
                    bypass_cache=True,
                    max_pairs=None,
                    split="val",
                )
                val_pck = val_results["per_keypoint"]["PCK@0.1"]
                print(f"  epoch {epoch + 1} val PCK@0.1: {val_pck:.2f}%")

                # Save to local SSD only when this epoch beats the previous best.
                if val_pck > best_val_pck:
                    best_val_pck = val_pck
                    torch.save(backbone.state_dict(), local_ckpt)
                    print(f"  New best checkpoint (val PCK@0.1={val_pck:.2f}%) saved locally: {local_ckpt}")
                else:
                    print(f"  No improvement (best so far: {best_val_pck:.2f}%) — checkpoint not updated.")

                backbone.train()

            print(f"  Training done. Best val PCK@0.1: {best_val_pck:.2f}%")

            # Single Drive write after all epochs are done (10–50× faster than per-epoch).
            os.makedirs(cfg["checkpoint_dir"], exist_ok=True)
            shutil.copy2(local_ckpt, ckpt)
            print(f"  Best checkpoint synced to Drive: {ckpt}")

            checkpoint_paths[(model_key, N)] = ckpt
        else:
            checkpoint_paths[(model_key, N)] = None

    for cfg in runs:
        model_key = cfg["model_key"]
        N = cfg["unfreeze_n"]

        reload_pretrained(model_key)
        if N > 0:
            ckpt = checkpoint_paths[(model_key, N)]
            if ckpt is None or not os.path.exists(ckpt):
                raise FileNotFoundError(f"Missing checkpoint for {model_key} N={N}: {ckpt}")
            backbone = _get_backbone(model_key)
            backbone.load_state_dict(torch.load(ckpt, map_location="cpu"))
            backbone.to(device)

        _get_backbone(model_key).eval()

        # ── Full val evaluation (used by Stage 3 to pick best N without touching test) ──
        val_results = evaluate_cached(
            model_key,
            use_softargmax=False,
            save_csv=False,
            bypass_cache=True,
            max_pairs=None,
            split="val",
        )
        val_pck = val_results["per_keypoint"]
        val_row = {
            "model_key":       model_key,
            "N":               N,
            "lr":              cfg["lr"],
            "epochs":          cfg["epochs"] if N > 0 else 0,
            "max_train_pairs": cfg["max_train_pairs"] if N > 0 else 0,
            "eval_pairs":      val_results["total_image_pairs"],
            "eval_scope":      "val",
            **val_pck,
        }
        _upsert_summary_csv(csv_path, [val_row])
        print(f"  Val row saved to: {csv_path}")

        # ── Full test evaluation (final reporting only — never used for model selection) ──
        results = evaluate_cached(
            model_key,
            use_softargmax=False,
            save_csv=True,
            bypass_cache=True,
            max_pairs=None,
        )
        pck = results["per_keypoint"]
        full_row = {
            "model_key":       model_key,
            "N":               N,
            "lr":              cfg["lr"],
            "epochs":          cfg["epochs"] if N > 0 else 0,
            "max_train_pairs": cfg["max_train_pairs"] if N > 0 else 0,
            "eval_pairs":      results["total_image_pairs"],
            "eval_scope":      "full_test",
            **pck,
        }
        full_rows.append(full_row)
        _upsert_summary_csv(csv_path, [full_row])
        print(f"  Full-test row saved to: {csv_path}")

    print(f"\nAll {len(runs)} run(s) complete. Val + full-test results saved to: {csv_path}")

    sync_features_to_drive()


def _upsert_summary_csv(csv_path, new_rows):
    """Upsert rows keyed by (model_key, N, eval_scope).

    Lets per-N evaluations be written incrementally without erasing other
    (model_key, N) rows already on disk, and lets any cell be re-run
    safely while preserving rows for other backbones / N values.
    """
    if not new_rows:
        return
    new_df = pd.DataFrame(new_rows)
    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        key_cols = ["model_key", "N", "eval_scope"]
        if all(c in existing.columns for c in key_cols):
            new_keys = set(zip(new_df["model_key"], new_df["N"], new_df["eval_scope"]))
            existing_keys = list(zip(
                existing["model_key"], existing["N"], existing["eval_scope"]
            ))
            keep = [k not in new_keys for k in existing_keys]
            existing = existing[keep]
        combined = pd.concat([existing, new_df], ignore_index=True)
    else:
        combined = new_df
    combined.to_csv(csv_path, index=False)


In [14]:
# Stage 2 — fine-tune DINOv2 ViT-L/14.
# Runs the N ∈ {1, 2, 4} configurations and refreshes this backbone's rows in the summary CSVs.

_MODEL_KEY = "dinov2_vitl14"
_RUNS = []
for _N in [1, 2, 4]:
    _ckpt = os.path.join(_BASE["checkpoint_dir"], f"{_MODEL_KEY}_last{_N}.pth")
    if os.path.exists(_ckpt):
        print(f"Skipping {_MODEL_KEY} N={_N} because checkpoint already exists.")
        continue
    _RUNS.append({"model_key": _MODEL_KEY, "unfreeze_n": _N, **_BASE})

print(f"Pending runs for {_MODEL_KEY}: {len(_RUNS)}")
run_finetune_for_backbone(_RUNS)



In [15]:
# Stage 2 — fine-tune DINOv3 ViT-B/16.
# Runs the N ∈ {1, 2, 4} configurations and refreshes this backbone's rows in the summary CSVs.

_MODEL_KEY = "dinov3_vitb16"
_RUNS = []
for _N in [1, 2, 4]:
    _ckpt = os.path.join(_BASE["checkpoint_dir"], f"{_MODEL_KEY}_last{_N}.pth")
    if os.path.exists(_ckpt):
        print(f"Skipping {_MODEL_KEY} N={_N} because checkpoint already exists.")
        continue
    _RUNS.append({"model_key": _MODEL_KEY, "unfreeze_n": _N, **_BASE})

print(f"Pending runs for {_MODEL_KEY}: {len(_RUNS)}")
run_finetune_for_backbone(_RUNS)



In [16]:
# Stage 2 — fine-tune SAM ViT-B image encoder.
# Runs the N ∈ {1, 2, 4} configurations and refreshes this backbone's rows in the summary CSVs.

_MODEL_KEY = "sam_vit_b_res1024"
_RUNS = []
for _N in [1, 2, 4]:
    _ckpt = os.path.join(_BASE["checkpoint_dir"], f"{_MODEL_KEY}_last{_N}.pth")
    if os.path.exists(_ckpt):
        print(f"Skipping {_MODEL_KEY} N={_N} because checkpoint already exists.")
        continue
    _RUNS.append({"model_key": _MODEL_KEY, "unfreeze_n": _N, **_BASE})

print(f"Pending runs for {_MODEL_KEY}: {len(_RUNS)}")
run_finetune_for_backbone(_RUNS)



# Stage 3 — Better prediction rule: window soft-argmax

In Stages 1 and 2, the final correspondence is obtained by a plain argmax over the cosine-similarity map. This snaps predictions to the centre of a single patch and can be unstable when neighbouring patches have similar scores.

Window soft-argmax first finds the peak cell, extracts a small window around it, applies a temperature-scaled softmax inside that local window, and returns the expected coordinate. This produces a sub-patch prediction while still staying near the strongest match.

In this stage, the notebook loads the best fine-tuned checkpoint for each backbone, sweeps temperature/window-size combinations on the validation split, and evaluates the selected configuration on the test split.


In [17]:
# Stage 3 — select the best fine-tuned checkpoint per backbone.
# Uses validation PCK@0.1 from Stage 2 to choose N without looking at the test split.

import os
import json
import math
import shutil
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm.auto import tqdm

# ── Resolve best N per backbone from Stage 2 CSV ──────────────────────────────
FINETUNE_CSV = os.path.join(RESULTS_ROOT, "summary_finetune.csv")

# These are the checkpoint filenames written by run_finetune_for_backbone
CKPT_DIR = os.path.join(MYDRIVE, "checkpoints", "finetuned")

BEST_CKPTS = {}  # model_key -> {"N": int, "ckpt_path": str}

if os.path.exists(FINETUNE_CSV):
    ft_df = pd.read_csv(FINETUNE_CSV)
    # Use validation rows, not full-test rows, to pick best N and avoid test-set leakage.
    # Stage 2 now writes eval_scope=='val' rows alongside eval_scope=='full_test' rows.
    val_df = ft_df[(ft_df["eval_scope"] == "val") & (ft_df["N"] > 0)].copy()
    if not val_df.empty:
        # For each backbone pick the N with highest VAL PCK@0.1
        for mk in val_df["model_key"].unique():
            sub = val_df[val_df["model_key"] == mk]
            best_row = sub.loc[sub["PCK@0.1"].idxmax()]
            N = int(best_row["N"])
            ckpt = os.path.join(CKPT_DIR, f"{mk}_last{N}.pth")
            if os.path.exists(ckpt):
                BEST_CKPTS[mk] = {"N": N, "ckpt_path": ckpt, "stage2_pck01": float(best_row["PCK@0.1"])}
                print(f"  {mk}: best N={N}  (val PCK@0.1 = {best_row['PCK@0.1']:.2f}%)  ✓ checkpoint found")
            else:
                print(f"  {mk}: best N={N} — checkpoint NOT found at {ckpt}")
    else:
        print("No val rows found in summary_finetune.csv — falling back to frozen features.")
else:
    print(f"summary_finetune.csv not found at {FINETUNE_CSV}. Will evaluate frozen models (N=0).")

# Fall back to frozen evaluation for any missing backbone
for mk in MODEL_SPECS:
    if mk not in BEST_CKPTS:
        BEST_CKPTS[mk] = {"N": 0, "ckpt_path": None, "stage2_pck01": None}
        print(f"  {mk}: using frozen baseline (N=0)")

print("\nBest checkpoints:", BEST_CKPTS)


  mae_vit_h14: best N=2  (val PCK@0.1 = 2.04%)  ✓ checkpoint found
  dinov2_vitl14: using frozen baseline (N=0)
  dinov3_vitb16: using frozen baseline (N=0)
  sam_vit_b_res1024: using frozen baseline (N=0)

Best checkpoints: {'mae_vit_h14': {'N': 2, 'ckpt_path': '/content/drive/MyDrive/checkpoints/finetuned/mae_vit_h14_last2.pth', 'stage2_pck01': 2.0410763428804404}, 'dinov2_vitl14': {'N': 0, 'ckpt_path': None, 'stage2_pck01': None}, 'dinov3_vitb16': {'N': 0, 'ckpt_path': None, 'stage2_pck01': None}, 'sam_vit_b_res1024': {'N': 0, 'ckpt_path': None, 'stage2_pck01': None}}


In [18]:
# Stage 3 — checkpoint loading helper.
# Reloads pretrained weights and overlays the selected fine-tuned checkpoint when available.

def load_best_checkpoint(model_key):
    """
    Reload pretrained weights and, if a fine-tuned checkpoint exists
    (N > 0), overlay the best Stage 2 weights on top.
    After this call the global model object is ready for inference.
    """
    info = BEST_CKPTS[model_key]
    reload_pretrained(model_key)            # reset to pretrained
    backbone = _get_backbone(model_key)
    if info["N"] > 0 and info["ckpt_path"] and os.path.exists(info["ckpt_path"]):
        state = torch.load(info["ckpt_path"], map_location="cpu")
        backbone.load_state_dict(state)
        print(f"  Loaded fine-tuned weights (N={info['N']}): {info['ckpt_path']}")
    else:
        print(f"  Using frozen (pretrained) weights for {model_key}")
    backbone.to(device).eval()


print("load_best_checkpoint() defined.")


load_best_checkpoint() defined.


In [19]:
# Stage 3 — manual BEST_CKPTS override.
# Use this only when validation rows are missing from the Stage 2 summary but the best checkpoints are already known.

CKPT_DIR = os.path.join(MYDRIVE, "checkpoints", "finetuned")

BEST_CKPTS = {
    "dinov2_vitl14":     {"N": 4, "ckpt_path": os.path.join(CKPT_DIR, "dinov2_vitl14_last4.pth")},
    "dinov3_vitb16":     {"N": 2, "ckpt_path": os.path.join(CKPT_DIR, "dinov3_vitb16_last2.pth")},
    "sam_vit_b_res1024": {"N": 2, "ckpt_path": os.path.join(CKPT_DIR, "sam_vit_b_res1024_last2.pth")},
}

for mk, info in BEST_CKPTS.items():
    exists = os.path.exists(info["ckpt_path"])
    print(f"{mk}: N={info['N']}  checkpoint={'FOUND' if exists else 'MISSING'}")

dinov2_vitl14: N=4  checkpoint=FOUND
dinov3_vitb16: N=2  checkpoint=FOUND
sam_vit_b_res1024: N=2  checkpoint=FOUND


In [20]:
# Stage 3 — window soft-argmax validation sweep.
# Computes similarity maps once per backbone and evaluates all temperature/window combinations on the validation split.

TEMP_GRID   = [0.001, 0.005, 0.01, 0.05]   # softmax temperature τ
WINDOW_GRID = [3, 5, 7, 11]                 # window size w

S3_SWEEP_RESULTS = {}  # model_key -> list of dicts


def _compute_val_sim_maps(model_key):
    """
    Single pass over the val split with the CURRENT model weights.
    For every valid keypoint in every pair we compute the cosine-
    similarity map between the source keypoint feature and the
    entire target feature grid, then store the map on CPU together
    with the ground-truth target pixel and normalisation factor.

    Returns a list of dicts, one per valid keypoint:
        sim          – (fh, fw) float32 CPU tensor
        tx, ty       – ground-truth target pixel coordinates
        norm_factor  – max(bbox_w, bbox_h) used for PCK
        fw, fh       – feature grid dimensions
        trg_w, trg_h – target image pixel dimensions
    """
    _device = 'cuda' if torch.cuda.is_available() else 'cpu'
    pair_files = list_pair_files(split="val")
    records = []

    for pair_path in tqdm(pair_files, desc=f'[{model_key}] sim-maps'):
        with open(pair_path, 'r') as fp:
            ann = json.load(fp)

        category = ann['category']
        src_img_path = os.path.join(JPEG_ROOT, category, ann['src_imname'])
        trg_img_path = os.path.join(JPEG_ROOT, category, ann['trg_imname'])

        src_pil = Image.open(src_img_path).convert('RGB')
        trg_pil = Image.open(trg_img_path).convert('RGB')
        src_w, src_h = src_pil.size
        trg_w, trg_h = trg_pil.size

        # ONE model forward pass per image (fine-tuned weights, bypass cache)
        raw_src = compute_feature(model_key, src_img_path)
        raw_trg = compute_feature(model_key, trg_img_path)
        f_src = F.normalize(raw_src.float(), dim=0).to(_device)
        f_trg = F.normalize(raw_trg.float(), dim=0).to(_device)

        fh, fw = f_src.shape[1], f_src.shape[2]

        trg_bbox    = ann['trg_bndbox']
        norm_factor = max(float(trg_bbox[2] - trg_bbox[0]),
                          float(trg_bbox[3] - trg_bbox[1]))

        src_kps = ann['src_kps']
        trg_kps = ann['trg_kps']
        kp_indices = range(len(src_kps)) if isinstance(src_kps, list) else src_kps.keys()

        for idx in kp_indices:
            p_src = _parse_xy(src_kps[idx])
            p_trg = _parse_xy(trg_kps[idx])
            if p_src is None or p_trg is None:
                continue

            sx, sy = p_src
            tx, ty = p_trg

            feat_x = max(0, int(min(sx / src_w * fw, fw - 1)))
            feat_y = max(0, int(min(sy / src_h * fh, fh - 1)))
            target_feat = f_src[:, feat_y, feat_x]  # (D,)

            # Cosine similarity map — identical to evaluate_cached
            sim = torch.einsum('c,chw->hw', target_feat, f_trg)  # (fh, fw)

            records.append({
                'sim':         sim.cpu(),   # move to CPU to free GPU memory
                'tx': tx,      'ty': ty,
                'norm_factor': norm_factor,
                'fw': fw,      'fh': fh,
                'trg_w': trg_w,'trg_h': trg_h,
            })

    return records


for model_key in MODEL_SPECS:
    print(f"\n{'='*60}")
    print(f"  Soft-argmax val sweep: {model_key}")
    print(f"{'='*60}")

    load_best_checkpoint(model_key)

    # ── Phase 1: ONE forward pass, cache all sim maps ─────────────────────
    print(f"  Phase 1: computing similarity maps (1 pass, no repeats)…")
    sim_records = _compute_val_sim_maps(model_key)
    print(f"  Cached {len(sim_records)} keypoint sim maps in RAM.")

    # ── Phase 2: sweep (temp, window) — zero model calls ─────────────────
    _device = 'cuda' if torch.cuda.is_available() else 'cpu'
    n_configs = len(TEMP_GRID) * len(WINDOW_GRID)
    print(f"  Phase 2: sweeping {n_configs} (temp, window) configs over cached maps…")

    rows = []
    for temp in TEMP_GRID:
        for win in WINDOW_GRID:
            correct_kps = {t: 0 for t in PCK_THRESHOLDS}
            total_kps   = 0

            for rec in sim_records:
                sim          = rec['sim'].to(_device)
                fw, fh       = rec['fw'], rec['fh']
                trg_w, trg_h = rec['trg_w'], rec['trg_h']
                tx, ty       = rec['tx'], rec['ty']
                norm_factor  = rec['norm_factor']

                # soft-argmax — same function used in evaluate_cached
                px_feat, py_feat = softargmax_2d(sim, temperature=temp, window_size=win)
                pred_x = ((px_feat.item() + 0.5) / fw) * trg_w
                pred_y = ((py_feat.item() + 0.5) / fh) * trg_h

                dist = math.sqrt((pred_x - tx) ** 2 + (pred_y - ty) ** 2)
                total_kps += 1
                for t in PCK_THRESHOLDS:
                    if dist <= t * norm_factor:
                        correct_kps[t] += 1

            pck_vals = {
                f'PCK@{t}': (correct_kps[t] / total_kps * 100.0)
                            if total_kps > 0 else 0.0
                for t in PCK_THRESHOLDS
            }
            row = {'model_key': model_key, 'temperature': temp,
                   'window_size': win, **pck_vals}
            rows.append(row)
            print(
                f"  temp={temp:<6}  win={win:<3}  "
                + "  ".join(f"PCK@{t}={pck_vals[f'PCK@{t}']:.2f}%" for t in PCK_THRESHOLDS)
            )

    S3_SWEEP_RESULTS[model_key] = rows
    sweep_path = os.path.join(RESULTS_ROOT, f"{model_key}_stage3_val_sweep.csv")
    pd.DataFrame(rows).to_csv(sweep_path, index=False)
    print(f"  Saved val sweep: {sweep_path}")

    # Free sim maps before loading the next backbone
    del sim_records

print("\nVal sweep complete for all backbones.")



In [21]:
# Stage 3 — final test comparison.
# Evaluates the selected checkpoint with argmax and with the best validation soft-argmax configuration.

S3_BEST_CFG   = {}   # model_key -> best (temp, win)
S3_TEST_ROWS  = []   # rows for the summary CSV

for model_key in MODEL_SPECS:
    rows = S3_SWEEP_RESULTS.get(model_key)
    if not rows:
        # Try loading from disk (if sweep was run in a previous session)
        sweep_path = os.path.join(RESULTS_ROOT, f"{model_key}_stage3_val_sweep.csv")
        if os.path.exists(sweep_path):
            rows = pd.read_csv(sweep_path).to_dict("records")
        else:
            print(f"  No val sweep data for {model_key} — skipping.")
            continue

    # Best config = highest val PCK@0.1
    best = max(rows, key=lambda r: r["PCK@0.1"])
    best_temp = best["temperature"]
    best_win  = int(best["window_size"])
    S3_BEST_CFG[model_key] = {"temperature": best_temp, "window_size": best_win}
    print(f"\n{model_key}: best val config -> temp={best_temp}  win={best_win}  "
          f"(val PCK@0.1={best['PCK@0.1']:.2f}%)")

    load_best_checkpoint(model_key)

    # ── Argmax on best checkpoint (test) ──────────────────────────────────────
    print(f"  Running argmax test eval ...")
    res_argmax = evaluate_cached(
        model_key,
        use_softargmax=False,
        save_csv=True,
        bypass_cache=True,
        max_pairs=None,
        split="test",
    )
    pck_argmax = res_argmax["per_keypoint"]

    # ── Soft-argmax on best checkpoint + best hyper-params (test) ─────────────
    print(f"  Running soft-argmax test eval (temp={best_temp}, win={best_win}) ...")
    res_sa = evaluate_cached(
        model_key,
        use_softargmax=True,
        softargmax_temp=best_temp,
        softargmax_window=best_win,
        save_csv=True,
        bypass_cache=True,
        max_pairs=None,
        split="test",
    )
    pck_sa = res_sa["per_keypoint"]

    for mode, pck in [("argmax", pck_argmax), ("softargmax", pck_sa)]:
        S3_TEST_ROWS.append({
            "model_key": model_key,
            "N_finetuned": BEST_CKPTS[model_key]["N"],
            "mode": mode,
            "temperature": best_temp if mode == "softargmax" else None,
            "window_size": best_win if mode == "softargmax" else None,
            **pck,
        })

# Save summary
s3_summary_path = os.path.join(RESULTS_ROOT, "stage3_summary.csv")
s3_df = pd.DataFrame(S3_TEST_ROWS)
s3_df.to_csv(s3_summary_path, index=False)
print(f"\nSaved Stage 3 summary: {s3_summary_path}")
print(s3_df.to_string(index=False))



In [22]:
# Stage 3 — display argmax vs soft-argmax results.
# Loads the Stage 3 summary CSV, prints threshold-wise scores and deltas, and syncs results to Drive.

s3_df = pd.read_csv(os.path.join(RESULTS_ROOT, "stage3_summary.csv"))

print("\n" + "="*90)
print("STAGE 3 RESULTS — Argmax vs Window Soft-Argmax (full test split)")
print("="*90)

header = f"{'Model':<28} {'Mode':<14} " + "  ".join(f"PCK@{t}" for t in PCK_THRESHOLDS)
print(header)
print("-" * 90)

for mk in s3_df["model_key"].unique():
    sub = s3_df[s3_df["model_key"] == mk]
    for mode in ["argmax", "softargmax"]:
        row = sub[sub["mode"] == mode]
        if row.empty:
            continue
        row = row.iloc[0]
        label = f"{mk} [{mode}]"
        pck_str = "  ".join(f"{row[f'PCK@{t}']:>8.2f}%" for t in PCK_THRESHOLDS)
        print(f"{label:<42} {pck_str}")

    # Print delta row
    arg_row = sub[sub["mode"] == "argmax"]
    sa_row  = sub[sub["mode"] == "softargmax"]
    if not arg_row.empty and not sa_row.empty:
        arg_row = arg_row.iloc[0]
        sa_row  = sa_row.iloc[0]
        delta_str = "  ".join(
            f"{sa_row[f'PCK@{t}'] - arg_row[f'PCK@{t}']:>+8.2f}%"
            for t in PCK_THRESHOLDS
        )
        cfg = S3_BEST_CFG.get(mk, {})
        print(f"{'  Δ (soft-argmax − argmax)':<42} {delta_str}"
              f"   [temp={cfg.get('temperature','?')}, win={cfg.get('window_size','?')}]")
    print()

print("="*90)
print("\nInterpretation:")
print("  • Positive Δ at PCK@0.05 / 0.1 indicates soft-argmax improves fine-grained accuracy.")
print("  • Benefits are typically larger at tight thresholds (0.05) where sub-pixel")
print("    precision matters most.")

# Sync new results to Drive
sync_features_to_drive()
print("\nAll Stage 3 results synced to Drive.")



# Stage 4 — Mandatory extension: PF-Pascal generalisation

PF-Pascal is a cross-domain semantic-correspondence benchmark built on Pascal VOC 2012. Unlike SPair-71k, which normalises PCK by the target bounding-box size, PF-Pascal uses **PCK@α = α · max(H, W)**.

This extension evaluates all three backbones under three conditions: frozen argmax, best fine-tuned checkpoint with argmax, and best fine-tuned checkpoint with the Stage 3 window soft-argmax configuration.

The goal is to check whether improvements learned on SPair-71k transfer to PF-Pascal without training on PF-Pascal itself.


In [23]:
# Stage 4 — prepare the PF-Pascal dataset.
# Downloads/copies the dataset, extracts it locally, builds per-category pair CSVs, and performs sanity checks.

import os, shutil, glob, scipy.io
import numpy as np
import pandas as pd

PFP_ZIP       = os.path.join(MYDRIVE, 'PF-dataset-PASCAL.zip')
PFP_DIR       = os.path.join(MYDRIVE, 'PF-dataset-PASCAL')
LOCAL_PFP_DIR = os.path.join(LOCAL_DATA_DIR, 'PF-dataset-PASCAL')

PFP_URL = (
    'https://www.di.ens.fr/willow/research/proposalflow/'
    'dataset/PF-dataset-PASCAL.zip'
)

PFP_CATEGORIES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]

# ── Step A: download to Drive if needed ─────────────────────
if not os.path.exists(PFP_ZIP) and not os.path.isdir(PFP_DIR):
    print("Downloading PF-Pascal ...")
    !wget -q --show-progress -O "{PFP_ZIP}" "{PFP_URL}"
    print("Download complete.")
else:
    print("PF-Pascal archive / directory already on Drive — skipping download.")

# ── Step B: copy/extract to local SSD ───────────────────────
if not os.path.isdir(LOCAL_PFP_DIR):
    if os.path.exists(PFP_ZIP):
        print(f"Extracting {PFP_ZIP} to {LOCAL_DATA_DIR} ...")
        !unzip -q "{PFP_ZIP}" -d "{LOCAL_DATA_DIR}"

        candidates = [
            p for p in glob.glob(os.path.join(LOCAL_DATA_DIR, 'PF-dataset*'))
            if os.path.isdir(p)
        ]

        if candidates:
            extracted_dir = candidates[0]
            if extracted_dir != LOCAL_PFP_DIR:
                os.rename(extracted_dir, LOCAL_PFP_DIR)

        print("Extraction complete.")

    elif os.path.isdir(PFP_DIR):
        print(f"Copying {PFP_DIR} from Drive to local SSD ...")
        shutil.copytree(PFP_DIR, LOCAL_PFP_DIR)
        print("Copy complete.")

    else:
        raise FileNotFoundError(
            "PF-Pascal not found. Put either PF-dataset-PASCAL.zip or "
            "PF-dataset-PASCAL folder in your Google Drive."
        )
else:
    print("PF-Pascal already on local SSD:", LOCAL_PFP_DIR)

# ── Step C: locate required folders/files ───────────────────
parse_mat_path = os.path.join(LOCAL_PFP_DIR, 'parsePascalVOC.mat')
ann_root       = os.path.join(LOCAL_PFP_DIR, 'Annotations')
img_root       = os.path.join(LOCAL_PFP_DIR, 'JPEGImages')
pairs_dir      = os.path.join(LOCAL_PFP_DIR, 'image_pairs')

if not os.path.exists(parse_mat_path):
    raise FileNotFoundError(f"Missing parsePascalVOC.mat: {parse_mat_path}")

if not os.path.isdir(ann_root):
    raise FileNotFoundError(f"Missing annotation folder: {ann_root}")

if not os.path.isdir(img_root):
    raise FileNotFoundError(f"Missing image folder: {img_root}")

# Force-recreate image_pairs to remove old numeric/wrong CSVs
if os.path.isdir(pairs_dir):
    shutil.rmtree(pairs_dir)
os.makedirs(pairs_dir, exist_ok=True)

# ── Step D: read parsePascalVOC.mat ─────────────────────────
raw = scipy.io.loadmat(parse_mat_path, squeeze_me=True, struct_as_record=False)
data_keys = [k for k in raw.keys() if not k.startswith('_')]
print("parsePascalVOC.mat keys:", data_keys)

if 'PascalVOC' not in raw:
    raise RuntimeError("Expected key 'PascalVOC' inside parsePascalVOC.mat")

pv = raw['PascalVOC']

if not hasattr(pv, 'pair'):
    raise RuntimeError("Expected field 'pair' inside PascalVOC struct")

pair_array = pv.pair

# Convert numeric Pascal VOC ID to filename:
#   2007006212 -> 2007_006212.jpg
def voc_id_to_filename(voc_id):
    s = str(int(round(float(voc_id))))
    return s[:4] + '_' + s[4:] + '.jpg'

# ── Step E: write one pair CSV per category ─────────────────
n_written = 0

for cat_idx, category in enumerate(PFP_CATEGORIES):
    pair_data = np.array(pair_array[cat_idx], dtype=float)

    if pair_data.ndim == 1:
        pair_data = pair_data.reshape(1, -1)

    if pair_data.shape[1] != 2 and pair_data.shape[0] == 2:
        pair_data = pair_data.T

    rows = []
    for i in range(pair_data.shape[0]):
        src_name = voc_id_to_filename(pair_data[i, 0])
        trg_name = voc_id_to_filename(pair_data[i, 1])
        rows.append((src_name, trg_name))

    csv_path = os.path.join(pairs_dir, f'{category}.csv')
    pd.DataFrame(rows).to_csv(csv_path, index=False, header=False)
    n_written += 1

print(f"image_pairs CSVs written: {n_written} / {len(PFP_CATEGORIES)}")

# ── Step F: sanity checks ───────────────────────────────────
n_mat_recursive = len(glob.glob(os.path.join(ann_root, '**', '*.mat'), recursive=True))
n_pair_csvs     = len(glob.glob(os.path.join(pairs_dir, '*.csv')))
n_images        = len(glob.glob(os.path.join(img_root, '*.jpg')))

print()
print("PF-Pascal sanity check")
print("----------------------")
print("LOCAL_PFP_DIR          :", LOCAL_PFP_DIR)
print("Annotation root        :", ann_root)
print("Recursive .mat files   :", n_mat_recursive)
print("Pair CSV files         :", n_pair_csvs)
print("JPEG images            :", n_images)

if n_mat_recursive == 0:
    raise RuntimeError(
        "No annotation .mat files found recursively under Annotations/. "
        "Your annotation folder is missing or copied to the wrong place."
    )

if n_pair_csvs == 0:
    raise RuntimeError("No image-pair CSVs were generated.")

first_csv = os.path.join(pairs_dir, f'{PFP_CATEGORIES[0]}.csv')
print()
print(f"First 5 rows of {PFP_CATEGORIES[0]}.csv:")
print(pd.read_csv(first_csv, header=None).head().to_string(index=False))

print()
print("First 10 annotation .mat files:")
for p in glob.glob(os.path.join(ann_root, '**', '*.mat'), recursive=True)[:10]:
    print(" ", os.path.relpath(p, ann_root))

print()
print("PF-Pascal ready.")


In [24]:
# Stage 4 — PF-Pascal loader and evaluator.
# Parses per-image annotations and evaluates PCK with max(target_H, target_W) normalisation.

import os, glob, math
import scipy.io
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

PFP_CATEGORIES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]

LOCAL_PFP_FEATURE_ROOT = '/content/features/pfpascal'
DRIVE_PFP_FEATURE_ROOT = os.path.join(MYDRIVE, 'features', 'pfpascal')
os.makedirs(LOCAL_PFP_FEATURE_ROOT, exist_ok=True)


def _pfp_image_id(image_name):
    """
    Convert '2008_000602.jpg' -> '2008_000602'.
    """
    return os.path.splitext(os.path.basename(str(image_name).strip()))[0]


def _find_pfpascal_annotation(category, image_name):
    """
    Find annotation file for one image.

    Expected:
        Annotations/{category}/{image_id}.mat

    Also includes recursive fallback, just in case the folder layout differs.
    """
    ann_root = os.path.join(LOCAL_PFP_DIR, 'Annotations')
    image_id = _pfp_image_id(image_name)

    direct_path = os.path.join(ann_root, category, image_id + '.mat')
    if os.path.exists(direct_path):
        return direct_path

    fallback = glob.glob(
        os.path.join(ann_root, '**', image_id + '.mat'),
        recursive=True
    )

    if len(fallback) > 0:
        return fallback[0]

    raise FileNotFoundError(
        f"Missing annotation for category='{category}', image='{image_name}'. "
        f"Tried: {direct_path}"
    )


def _load_pfpascal_annotation(category, image_name):
    """
    Load one PF-Pascal image-level annotation.

    Expected .mat keys:
        kps      : shape usually (16, 2)
        bbox     : optional
        imsize   : optional
        class    : optional

    Returns:
        dict with kps and valid mask.
    """
    mat_path = _find_pfpascal_annotation(category, image_name)
    raw = scipy.io.loadmat(mat_path, squeeze_me=True, struct_as_record=False)

    if 'kps' not in raw:
        raise KeyError(f"'kps' not found in annotation file: {mat_path}")

    kps = np.asarray(raw['kps'], dtype=np.float32)

    # Normalise possible shapes to (n_kps, 2)
    if kps.ndim == 1:
        kps = kps.reshape(-1, 2)

    if kps.ndim == 2:
        if kps.shape[0] == 2 and kps.shape[1] != 2:
            kps = kps.T
    else:
        raise ValueError(f"Unexpected kps shape {kps.shape} in {mat_path}")

    # Valid keypoints: finite and positive.
    # PF-Pascal annotations are MATLAB-style, so valid coordinates are usually > 0.
    valid = np.isfinite(kps).all(axis=1)
    valid &= (kps[:, 0] > 0)
    valid &= (kps[:, 1] > 0)

    # Optional per-keypoint visibility fields, if present in some versions
    for vis_key in ['kps_vis', 'kps_valid', 'vis']:
        if vis_key in raw:
            vis = np.asarray(raw[vis_key]).astype(float).reshape(-1)
            n = min(len(valid), len(vis))
            valid[:n] &= (vis[:n] > 0)
            break

    bbox = np.asarray(raw.get('bbox', []), dtype=np.float32).reshape(-1)
    imsize = np.asarray(raw.get('imsize', []), dtype=np.float32).reshape(-1)

    return {
        'path': mat_path,
        'kps': kps,
        'valid': valid,
        'bbox': bbox,
        'imsize': imsize,
    }


def _load_pfpascal_pairs(category):
    """
    Load image pairs for one PF-Pascal category.

    Expected CSV:
        image_pairs/{category}.csv

    Format:
        src_filename, trg_filename
    """
    csv_path = os.path.join(LOCAL_PFP_DIR, 'image_pairs', f'{category}.csv')

    if not os.path.exists(csv_path):
        alt = os.path.join(LOCAL_PFP_DIR, 'image_pairs', f'test_pairs_{category}.csv')
        if os.path.exists(alt):
            csv_path = alt
        else:
            raise FileNotFoundError(
                f"Pair CSV not found for category '{category}'. "
                f"Tried {csv_path} and {alt}"
            )

    df = pd.read_csv(csv_path, header=None, sep=None, engine='python', dtype=str)

    if df.shape[1] < 2:
        raise ValueError(
            f"Pair CSV has fewer than 2 columns: {csv_path}"
        )

    pairs = []
    for _, row in df.iterrows():
        src_name = str(row.iloc[0]).strip()
        trg_name = str(row.iloc[1]).strip()

        if not src_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            src_name += '.jpg'
        if not trg_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            trg_name += '.jpg'

        pairs.append((src_name, trg_name))

    return pairs


def _find_pfpascal_image(image_name):
    """
    Find image file.

    Expected:
        JPEGImages/{image_name}

    Includes recursive fallback for alternative layouts.
    """
    img_root = os.path.join(LOCAL_PFP_DIR, 'JPEGImages')
    image_name = os.path.basename(str(image_name).strip())

    direct_path = os.path.join(img_root, image_name)
    if os.path.exists(direct_path):
        return direct_path

    fallback = glob.glob(
        os.path.join(img_root, '**', image_name),
        recursive=True
    )

    if len(fallback) > 0:
        return fallback[0]

    return None


def pfp_feature_path(model_key, img_path):
    """
    Cache path for a PF-Pascal image feature.
    """
    img_root = os.path.join(LOCAL_PFP_DIR, 'JPEGImages')
    rel = os.path.relpath(img_path, img_root)
    rel_no_ext = os.path.splitext(rel)[0]
    return os.path.join(LOCAL_PFP_FEATURE_ROOT, model_key, rel_no_ext + '.pt')


def _pfp_load_or_compute(model_key, img_path):
    """
    Load cached PF-Pascal feature or compute and cache it.
    """
    cache_path = pfp_feature_path(model_key, img_path)

    if os.path.exists(cache_path):
        return torch.load(cache_path, map_location='cpu')

    feat = compute_feature(model_key, img_path)

    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    torch.save(feat, cache_path)

    return feat


def evaluate_pfpascal(
    model_key,
    use_softargmax=False,
    softargmax_temp=0.01,
    softargmax_window=5,
    thresholds=None,
    bypass_cache=False,
    save_csv=True,
    label=None,
    max_pairs=None,
):
    """
    Evaluate semantic correspondence on PF-Pascal.

    PCK normalisation:
        max(target_height, target_width)

    Tests out-of-distribution generalisation:
        SPair-71k fine-tuned models evaluated on PF-Pascal without any PF-Pascal training.
    """
    if thresholds is None:
        thresholds = PCK_THRESHOLDS

    _device = 'cuda' if torch.cuda.is_available() else 'cpu'

    correct_kps    = {t: 0 for t in thresholds}
    total_kps      = 0
    per_image_rows = []

    skipped_missing_images = 0
    skipped_missing_anns   = 0

    for category in tqdm(PFP_CATEGORIES, desc=f'PF-Pascal eval [{model_key}]'):

        try:
            pairs = _load_pfpascal_pairs(category)
        except FileNotFoundError as e:
            print(f"Warning: skipping category '{category}': {e}")
            continue

        if max_pairs is not None:
            pairs = pairs[:max_pairs]

        for src_name, trg_name in pairs:

            src_path = _find_pfpascal_image(src_name)
            trg_path = _find_pfpascal_image(trg_name)

            if src_path is None or trg_path is None:
                skipped_missing_images += 1
                continue

            try:
                src_ann = _load_pfpascal_annotation(category, src_name)
                trg_ann = _load_pfpascal_annotation(category, trg_name)
            except FileNotFoundError:
                skipped_missing_anns += 1
                continue

            src_pil = Image.open(src_path).convert('RGB')
            trg_pil = Image.open(trg_path).convert('RGB')

            src_w, src_h = src_pil.size
            trg_w, trg_h = trg_pil.size

            norm_factor = max(trg_h, trg_w)

            if bypass_cache:
                raw_src = compute_feature(model_key, src_path)
                raw_trg = compute_feature(model_key, trg_path)
            else:
                raw_src = _pfp_load_or_compute(model_key, src_path)
                raw_trg = _pfp_load_or_compute(model_key, trg_path)

            f_src = F.normalize(raw_src.float(), dim=0).to(_device)
            f_trg = F.normalize(raw_trg.float(), dim=0).to(_device)

            _, fh, fw = f_src.shape

            src_kps = src_ann['kps']
            trg_kps = trg_ann['kps']

            valid = src_ann['valid'] & trg_ann['valid']
            n_kps = min(len(src_kps), len(trg_kps), len(valid))

            image_correct = {t: 0 for t in thresholds}
            image_total   = 0

            for ki in range(n_kps):

                if not valid[ki]:
                    continue

                sp = src_kps[ki]
                tp = trg_kps[ki]

                if np.any(np.isnan(sp)) or np.any(np.isnan(tp)):
                    continue

                # Convert MATLAB 1-based keypoint coords to Python 0-based coords
                sx, sy = float(sp[0]) - 1.0, float(sp[1]) - 1.0
                tx, ty = float(tp[0]) - 1.0, float(tp[1]) - 1.0

                if sx < 0 or sy < 0 or tx < 0 or ty < 0:
                    continue

                # Source image coordinate -> source feature grid coordinate
                feat_x = max(0, int(min(sx / src_w * fw, fw - 1)))
                feat_y = max(0, int(min(sy / src_h * fh, fh - 1)))

                target_feat = f_src[:, feat_y, feat_x]

                # Cosine similarity map over target features
                sim = torch.einsum('c,chw->hw', target_feat, f_trg)

                if use_softargmax:
                    px_feat, py_feat = softargmax_2d(
                        sim,
                        temperature=softargmax_temp,
                        window_size=softargmax_window
                    )

                    pred_x = ((px_feat.item() + 0.5) / fw) * trg_w
                    pred_y = ((py_feat.item() + 0.5) / fh) * trg_h

                else:
                    flat = sim.argmax().item()

                    pred_x = ((flat % fw + 0.5) / fw) * trg_w
                    pred_y = ((flat // fw + 0.5) / fh) * trg_h

                dist = math.sqrt((pred_x - tx) ** 2 + (pred_y - ty) ** 2)

                total_kps   += 1
                image_total += 1

                for t in thresholds:
                    if dist <= t * norm_factor:
                        correct_kps[t]   += 1
                        image_correct[t] += 1

            per_image_rows.append({
                'category':  category,
                'src_image': src_name,
                'trg_image': trg_name,
                'total_kps': image_total,
                **{
                    f'PCK@{t}': (
                        image_correct[t] / image_total * 100.0
                        if image_total > 0 else 0.0
                    )
                    for t in thresholds
                },
            })

    per_keypoint = {
        f'PCK@{t}': (
            correct_kps[t] / total_kps * 100.0
            if total_kps > 0 else 0.0
        )
        for t in thresholds
    }

    print()
    print("PF-Pascal evaluation finished")
    print("-----------------------------")
    print("Model                  :", model_key)
    print("Use softargmax          :", use_softargmax)
    print("Total evaluated pairs   :", len(per_image_rows))
    print("Total evaluated keypoints:", total_kps)
    print("Skipped missing images  :", skipped_missing_images)
    print("Skipped missing annots  :", skipped_missing_anns)

    print()
    print("PF-Pascal per-keypoint PCK:")
    for t in thresholds:
        print(f"  PCK@{t}: {per_keypoint[f'PCK@{t}']:.2f}%")

    if save_csv:
        tag = label or ('softargmax' if use_softargmax else 'argmax')

        csv1 = os.path.join(
            RESULTS_ROOT,
            f'pfpascal_{model_key}_{tag}_per_image.csv'
        )

        csv2 = os.path.join(
            RESULTS_ROOT,
            f'pfpascal_{model_key}_{tag}_per_keypoint.csv'
        )

        pd.DataFrame(per_image_rows).to_csv(csv1, index=False)

        pd.DataFrame([{
            'model_key': model_key,
            'dataset': 'PF-Pascal',
            'total_keypoints': total_kps,
            'total_image_pairs': len(per_image_rows),
            **per_keypoint,
        }]).to_csv(csv2, index=False)

        print()
        print("Saved:", csv1)
        print("Saved:", csv2)

    return {
        'model_key': model_key,
        'dataset': 'PF-Pascal',
        'use_softargmax': use_softargmax,
        'per_keypoint': per_keypoint,
        'per_image': per_image_rows,
        'total_keypoints': total_kps,
        'total_image_pairs': len(per_image_rows),
        'skipped_missing_images': skipped_missing_images,
        'skipped_missing_annotations': skipped_missing_anns,
    }


print("Fixed PF-Pascal image-level annotation loader and evaluate_pfpascal() defined.")


In [25]:
# Stage 4 — PF-Pascal smoke test.
# Runs a small DINOv2 frozen evaluation to validate paths, annotations, pair CSVs, and feature extraction before the full run.

reload_pretrained("dinov2_vitl14")
_get_backbone("dinov2_vitl14").to(device).eval()

test_res = evaluate_pfpascal(
    "dinov2_vitl14",
    use_softargmax=False,
    bypass_cache=True,   # safer first test: force fresh feature computation
    save_csv=False,
    max_pairs=2          # only 2 pairs per category, fast sanity check
)

smoke_total_keypoints = int(test_res.get("total_keypoints", 0))
smoke_total_pairs = int(test_res.get("total_image_pairs", 0))

print("\nPF-Pascal smoke-test summary")
print("-" * 40)
print(f"Evaluated image pairs : {smoke_total_pairs}")
print(f"Evaluated keypoints   : {smoke_total_keypoints}")
print("Per-keypoint PCK      :", test_res.get("per_keypoint", {}))

assert smoke_total_pairs > 0, (
    "PF-Pascal smoke test failed: no image pairs were evaluated. "
    "Check LOCAL_PFP_DIR, image_pairs CSV files, and image paths."
)
assert smoke_total_keypoints > 0, (
    "PF-Pascal smoke test failed: no valid keypoints were evaluated. "
    "Check .mat keypoint parsing and visibility handling."
)

PFP_SMOKE_TEST_PASSED = True
print("\n✅ PF-Pascal smoke test passed. You can now run the full Step 4 evaluation cell.")



In [26]:
# Stage 4 — rebuild S3_BEST_CFG from saved sweeps.
# Restores the best Stage 3 soft-argmax settings after a runtime restart.

import os
import pandas as pd

S3_BEST_CFG = {}

for model_key in MODEL_SPECS:
    sweep_path = os.path.join(RESULTS_ROOT, f"{model_key}_stage3_val_sweep.csv")

    if not os.path.exists(sweep_path):
        raise FileNotFoundError(
            f"Missing Stage 3 validation sweep file for {model_key}: {sweep_path}\n"
            "You need to run Stage 3 first, or make sure the Stage 3 CSV exists in RESULTS_ROOT."
        )

    sweep_df = pd.read_csv(sweep_path)

    if sweep_df.empty:
        raise RuntimeError(f"Stage 3 sweep file is empty for {model_key}: {sweep_path}")

    if "PCK@0.1" not in sweep_df.columns:
        raise RuntimeError(
            f"'PCK@0.1' column not found in {sweep_path}. "
            f"Available columns: {list(sweep_df.columns)}"
        )

    best_row = sweep_df.loc[sweep_df["PCK@0.1"].idxmax()]

    S3_BEST_CFG[model_key] = {
        "temperature": float(best_row["temperature"]),
        "window_size": int(best_row["window_size"]),
    }

    print(
        f"{model_key}: best soft-argmax config -> "
        f"temperature={S3_BEST_CFG[model_key]['temperature']}, "
        f"window_size={S3_BEST_CFG[model_key]['window_size']}"
    )

print("\nS3_BEST_CFG rebuilt successfully.")

dinov2_vitl14: best soft-argmax config -> temperature=0.05, window_size=7
dinov3_vitb16: best soft-argmax config -> temperature=0.05, window_size=7
sam_vit_b_res1024: best soft-argmax config -> temperature=0.05, window_size=11

S3_BEST_CFG rebuilt successfully.


In [27]:
# Stage 4 — full PF-Pascal evaluation.
# Compares frozen, fine-tuned argmax, and fine-tuned soft-argmax conditions for every backbone.

required_stage_vars = ["BEST_CKPTS", "S3_BEST_CFG", "MODEL_SPECS", "PCK_THRESHOLDS"]
missing_stage_vars = [name for name in required_stage_vars if name not in globals()]
if missing_stage_vars:
    raise RuntimeError(
        "Missing required variables from earlier stages: " + ", ".join(missing_stage_vars) +
        ". Re-run the setup and Stages 1–3 before running the PF-Pascal extension."
    )

if not globals().get("PFP_SMOKE_TEST_PASSED", False):
    raise RuntimeError(
        "Run STEP 4 — EXT CELL 2.5 first. The PF-Pascal smoke test must pass "
        "before launching the full PF-Pascal evaluation."
    )

PFP_RESULTS = []   # accumulates one dict per (model, condition)


for model_key in MODEL_SPECS:
    print(f"\n{'='*65}")
    print(f"  PF-Pascal evaluation: {model_key}")
    print(f"{'='*65}")

    # ── (A) FROZEN — reload pretrained, no fine-tuned weights ────────────────
    print("\n[A] Frozen baseline (argmax)")
    reload_pretrained(model_key)
    _get_backbone(model_key).to(device).eval()

    res_frozen = evaluate_pfpascal(
        model_key,
        use_softargmax=False,
        bypass_cache=False,   # use feature cache for speed
        save_csv=True,
        label='frozen_argmax',
    )
    PFP_RESULTS.append({
        'model_key':    model_key,
        'condition':    'frozen_argmax',
        'N_finetuned':  0,
        **res_frozen['per_keypoint'],
    })

    # ── (B) FINE-TUNED + ARGMAX ───────────────────────────────────────────────
    info = BEST_CKPTS.get(model_key, {})
    N    = info.get('N', 0)
    ckpt = info.get('ckpt_path')

    print(f"\n[B] Fine-tuned N={N} + argmax")
    reload_pretrained(model_key)
    if N > 0 and ckpt and os.path.exists(ckpt):
        backbone = _get_backbone(model_key)
        backbone.load_state_dict(torch.load(ckpt, map_location='cpu'))
        backbone.to(device).eval()
        print(f"  Loaded: {ckpt}")
    else:
        print(f"  No checkpoint found for N={N} — using pretrained weights.")

    res_ft = evaluate_pfpascal(
        model_key,
        use_softargmax=False,
        bypass_cache=True,    # fine-tuned weights differ from cached frozen features
        save_csv=True,
        label=f'ft_N{N}_argmax',
    )
    PFP_RESULTS.append({
        'model_key':   model_key,
        'condition':   f'ft_N{N}_argmax',
        'N_finetuned': N,
        **res_ft['per_keypoint'],
    })

    # ── (C) FINE-TUNED + SOFT-ARGMAX ─────────────────────────────────────────
    cfg       = S3_BEST_CFG.get(model_key, {'temperature': 0.01, 'window_size': 5})
    best_temp = cfg['temperature']
    best_win  = int(cfg['window_size'])
    print(f"\n[C] Fine-tuned N={N} + soft-argmax (temp={best_temp}, win={best_win})")

    res_sa = evaluate_pfpascal(
        model_key,
        use_softargmax=True,
        softargmax_temp=best_temp,
        softargmax_window=best_win,
        bypass_cache=True,
        save_csv=True,
        label=f'ft_N{N}_softargmax',
    )
    PFP_RESULTS.append({
        'model_key':    model_key,
        'condition':    f'ft_N{N}_softargmax',
        'N_finetuned':  N,
        'temperature':  best_temp,
        'window_size':  best_win,
        **res_sa['per_keypoint'],
    })

# ── Save combined summary ─────────────────────────────────────────────────────
pfp_summary_path = os.path.join(RESULTS_ROOT, 'pfpascal_extension_summary.csv')
pfp_df = pd.DataFrame(PFP_RESULTS)
pfp_df.to_csv(pfp_summary_path, index=False)
print(f"\nSaved PF-Pascal summary: {pfp_summary_path}")

# Sync PF-Pascal feature cache to Drive
print("\nSyncing PF-Pascal feature cache to Drive …")
os.makedirs(DRIVE_PFP_FEATURE_ROOT, exist_ok=True)
for root, dirs, files in os.walk(LOCAL_PFP_FEATURE_ROOT):
    rel      = os.path.relpath(root, LOCAL_PFP_FEATURE_ROOT)
    dst_root = os.path.join(DRIVE_PFP_FEATURE_ROOT, rel)
    os.makedirs(dst_root, exist_ok=True)
    for fn in files:
        src_f = os.path.join(root, fn)
        dst_f = os.path.join(dst_root, fn)
        if not os.path.exists(dst_f):
            shutil.copy2(src_f, dst_f)
print("Feature cache synced.")


In [28]:
# Stage 4 — display PF-Pascal results.
# Prints the extension summary table and a per-category PCK@0.1 breakdown for the selected fine-tuned condition.

pfp_df = pd.read_csv(os.path.join(RESULTS_ROOT, 'pfpascal_extension_summary.csv'))

print()
print("=" * 95)
print("STEP 4 RESULTS — PF-Pascal Semantic Correspondence (PCK@α · max(H,W))")
print("=" * 95)

COND_LABELS = {
    'frozen_argmax'   : 'Frozen       (argmax)',
}
for mk in pfp_df['model_key'].unique():
    for row in pfp_df[pfp_df['model_key'] == mk].itertuples():
        N = getattr(row, 'N_finetuned', 0)
        if 'ft' in row.condition and 'softargmax' not in row.condition:
            COND_LABELS[row.condition] = f'FT N={N}     (argmax)'
        elif 'ft' in row.condition and 'softargmax' in row.condition:
            COND_LABELS[row.condition] = f'FT N={N}     (soft-argmax)'

header = f"{'Model + Condition':<46} " + "   ".join(f"PCK@{t}" for t in PCK_THRESHOLDS)
print(header)
print("-" * 95)

for mk in pfp_df['model_key'].unique():
    sub = pfp_df[pfp_df['model_key'] == mk]
    for _, row in sub.iterrows():
        cond_label = COND_LABELS.get(row['condition'], row['condition'])
        label      = f"{mk}  [{cond_label}]"
        pck_vals   = "   ".join(f"{row[f'PCK@{t}']:>8.2f}%" for t in PCK_THRESHOLDS)
        print(f"{label:<64} {pck_vals}")
    print()

print("=" * 95)

# ── Cross-dataset Δ commentary ────────────────────────────────────────────────
print()
print("Cross-dataset generalisation (SPair-71k → PF-Pascal):")
print("  • Positive Δ between frozen and fine-tuned shows SPair-71k fine-tuning")
print("    generalises out-of-distribution to PF-Pascal.")
print("  • If fine-tuned PCK < frozen PCK, the backbone over-fitted to SPair-71k")
print("    annotation style (bounding-box normalised distances).")
print("  • Soft-argmax gains at PCK@0.05 mirror the SPair-71k pattern if the")
print("    generalisation is successful.")

# ── Per-category breakdown (optional) ─────────────────────────────────────────
print()
print("Per-category PCK@0.1 (fine-tuned best condition per backbone):")
print("-" * 80)

for model_key in pfp_df['model_key'].unique():
    # Pick the fine-tuned soft-argmax condition
    sub = pfp_df[pfp_df['model_key'] == model_key]
    ft_rows = sub[sub['condition'].str.contains('softargmax')]
    if ft_rows.empty:
        ft_rows = sub[sub['condition'].str.contains('ft')]
    if ft_rows.empty:
        continue

    # Read the per-image CSV for this condition
    chosen_cond = ft_rows.iloc[0]['condition']
    per_img_csv = os.path.join(
        RESULTS_ROOT, f'pfpascal_{model_key}_{chosen_cond}_per_image.csv'
    )
    if not os.path.exists(per_img_csv):
        continue

    per_img_df = pd.read_csv(per_img_csv)
    cat_pck = (
        per_img_df.groupby('category')['PCK@0.1']
        .mean()
        .sort_values(ascending=False)
    )
    print(f"\n{model_key} [{chosen_cond}]:")
    for cat, val in cat_pck.items():
        bar = '#' * int(val / 2)
        print(f"  {cat:<16} {val:>6.2f}%  {bar}")

print()
sync_features_to_drive()
print("All PF-Pascal results saved and synced.")


# Stage 4 — Mandatory extension: PF-Willow generalisation

PF-Willow is a cross-domain semantic-correspondence benchmark introduced with Proposal Flow. In this notebook, the dataset is loaded from the Proposal Flow-style category-folder structure, where each category contains image files and matching `.mat` annotation files.

This extension evaluates all three backbones under three conditions: frozen argmax, best fine-tuned checkpoint with argmax, and best fine-tuned checkpoint with the Stage 3 window soft-argmax configuration.

The goal is to check whether improvements learned on SPair-71k transfer to PF-Willow without training on PF-Willow itself.

In [31]:
# Stage 4 — prepare the PF-Willow dataset.
# Copies or extracts PF-Willow locally, ignores metadata folders, and locates the real category-folder root.

import os
import zipfile
import glob
import shutil

# Drive may contain either:
#   1) /content/drive/MyDrive/pf-willow.zip
#   2) /content/drive/MyDrive/pf-willow/PF-dataset/
PFW_ZIP_DRIVE = os.path.join(MYDRIVE, 'pf-willow.zip')
PFW_DIR_DRIVE = os.path.join(MYDRIVE, 'pf-willow', 'PF-dataset')

# Local extraction/copy folder inside Colab
LOCAL_PFW_EXTRACT_ROOT = os.path.join(LOCAL_DATA_DIR, 'pf-willow_extracted')

# Category folders exactly as shown in your Drive screenshots
PFW_CATEGORIES = [
    'car(G)', 'car(M)', 'car(S)',
    'duck(S)',
    'motorbike(G)', 'motorbike(M)', 'motorbike(S)',
    'winebottle(M)', 'winebottle(wC)', 'winebottle(woC)',
]


def find_pfwillow_root(search_root):
    """
    Find the real folder that directly contains the PF-Willow category folders.

    Important:
    - Ignores __MACOSX folders created by macOS zip files.
    - Ignores hidden metadata folders.
    - Requires the candidate folder to contain real image files and .mat annotations.
    """
    candidates = []

    for root, dirs, files in os.walk(search_root):

        # Ignore macOS metadata folders completely
        if '__MACOSX' in root.split(os.sep):
            continue

        # Ignore hidden folders such as .ipynb_checkpoints or AppleDouble metadata
        if any(part.startswith('.') for part in root.split(os.sep)):
            continue

        present = [
            cat for cat in PFW_CATEGORIES
            if os.path.isdir(os.path.join(root, cat))
        ]

        if len(present) >= 5:
            total_imgs = 0
            total_mats = 0

            for cat in present:
                cat_dir = os.path.join(root, cat)

                for ext in ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG']:
                    total_imgs += len(glob.glob(os.path.join(cat_dir, ext)))

                total_mats += len(glob.glob(os.path.join(cat_dir, '*.mat')))

            candidates.append((root, len(present), total_imgs, total_mats))

    if not candidates:
        raise RuntimeError(
            f"Could not find PF-Willow category folders under: {search_root}\n\n"
            "Run this command in a new Colab cell to inspect what was extracted:\n"
            "!find /content/data -maxdepth 6 -type d | sort | head -200"
        )

    # Prefer roots that contain real images and .mat files, not just folder names
    candidates.sort(
        key=lambda x: (x[2] > 0 and x[3] > 0, x[1], x[2], x[3]),
        reverse=True
    )

    best_root, n_cats, n_imgs, n_mats = candidates[0]

    if n_imgs == 0 or n_mats == 0:
        raise RuntimeError(
            "Found PF-Willow-like folders, but no real images or .mat files were found.\n\n"
            f"Best candidate was:\n{best_root}\n\n"
            "This usually means the code found only a metadata folder like __MACOSX.\n\n"
            "Run this command in a new Colab cell to inspect the extracted structure:\n"
            "!find /content/data -maxdepth 6 -type d | sort | head -200"
        )

    return best_root


# Clean only the local PF-Willow extraction folder to avoid stale wrong paths
if os.path.isdir(LOCAL_PFW_EXTRACT_ROOT):
    print('Removing old local PF-Willow extraction folder:', LOCAL_PFW_EXTRACT_ROOT)
    shutil.rmtree(LOCAL_PFW_EXTRACT_ROOT)

os.makedirs(LOCAL_PFW_EXTRACT_ROOT, exist_ok=True)


# Prefer the already-unzipped Drive folder if it exists.
# Otherwise use the zip file.
if os.path.isdir(PFW_DIR_DRIVE):
    print('PF-Willow folder found on Drive:', PFW_DIR_DRIVE)
    print('Copying PF-Willow folder from Drive to local Colab disk...')

    local_copy_target = os.path.join(LOCAL_PFW_EXTRACT_ROOT, 'PF-dataset')
    shutil.copytree(PFW_DIR_DRIVE, local_copy_target, dirs_exist_ok=True)

    LOCAL_PFW_DIR = find_pfwillow_root(LOCAL_PFW_EXTRACT_ROOT)

elif os.path.isfile(PFW_ZIP_DRIVE):
    print('PF-Willow zip found on Drive:', PFW_ZIP_DRIVE)
    print('Extracting', PFW_ZIP_DRIVE, 'to', LOCAL_PFW_EXTRACT_ROOT, '...')

    with zipfile.ZipFile(PFW_ZIP_DRIVE, 'r') as zf:
        zf.extractall(LOCAL_PFW_EXTRACT_ROOT)

    print('Extraction complete.')

    LOCAL_PFW_DIR = find_pfwillow_root(LOCAL_PFW_EXTRACT_ROOT)

else:
    raise FileNotFoundError(
        "Could not find PF-Willow dataset.\n\n"
        f"Expected either folder:\n"
        f"  {PFW_DIR_DRIVE}\n\n"
        f"or zip file:\n"
        f"  {PFW_ZIP_DRIVE}"
    )


print('\nDetected PF-Willow root:')
print(LOCAL_PFW_DIR)

if '__MACOSX' in LOCAL_PFW_DIR.split(os.sep):
    raise RuntimeError(
        'Wrong PF-Willow root detected: it points inside __MACOSX.\n'
        'This should never happen with the robust detector.'
    )

print('\nPF-Willow sanity check')
print('----------------------')

category_counts = {}
total_images = 0
total_mats = 0

for cat in PFW_CATEGORIES:
    cat_dir = os.path.join(LOCAL_PFW_DIR, cat)

    image_files = []
    for ext in ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG']:
        image_files.extend(glob.glob(os.path.join(cat_dir, ext)))

    mat_files = glob.glob(os.path.join(cat_dir, '*.mat'))

    ni = len(image_files)
    nm = len(mat_files)

    category_counts[cat] = (ni, nm)
    total_images += ni
    total_mats += nm

    print(f'{cat:16s} images={ni:4d}   mats={nm:4d}')

print('Total images:', total_images)
print('Total .mat annotations:', total_mats)

missing_or_empty = [
    cat for cat, (ni, nm) in category_counts.items()
    if ni == 0 or nm == 0
]

if missing_or_empty:
    raise RuntimeError(
        'Some PF-Willow category folders are missing images or .mat annotations: '
        + ', '.join(missing_or_empty)
        + '\n\nDetected root was:\n'
        + LOCAL_PFW_DIR
        + '\n\nRun this in a new Colab cell to inspect the actual structure:\n'
        + '!find /content/data -maxdepth 6 -type d | sort | head -200'
    )

print('\nPF-Willow dataset looks OK.')

Removing old local PF-Willow extraction folder: /content/data/pf-willow_extracted
PF-Willow zip found on Drive: /content/drive/MyDrive/pf-willow.zip
Extracting /content/drive/MyDrive/pf-willow.zip to /content/data/pf-willow_extracted ...
Extraction complete.

Detected PF-Willow root:
/content/data/pf-willow_extracted/PF-dataset

PF-Willow sanity check
----------------------
car(G)           images=  10   mats=  10
car(M)           images=  10   mats=  10
car(S)           images=  10   mats=  10
duck(S)          images=  10   mats=  10
motorbike(G)     images=  10   mats=  10
motorbike(M)     images=  10   mats=  10
motorbike(S)     images=  10   mats=  10
winebottle(M)    images=  10   mats=  10
winebottle(wC)   images=  10   mats=  10
winebottle(woC)  images=  10   mats=  10
Total images: 100
Total .mat annotations: 100

PF-Willow dataset looks OK.


In [32]:
# Stage 4 — PF-Willow loader and evaluator.
# Parses category-level image/keypoint files, creates evaluation pairs when needed, and computes PF-style PCK.

import os, glob, math
import scipy.io
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm


def _pfw_extract_keypoints_from_mat(mat_path):
    """Return keypoints as a float32 array with shape [K, 2] in 0-based pixels."""
    raw = scipy.io.loadmat(mat_path, squeeze_me=True, struct_as_record=False)

    preferred_keys = [
        'pts_coord', 'pts_coords', 'points', 'point', 'kps', 'kp',
        'keypoints', 'coords', 'coordinates', 'ann', 'annotation'
    ]

    candidates = []
    for key in preferred_keys + [k for k in raw.keys() if not k.startswith('__')]:
        if key not in raw:
            continue
        arr = np.asarray(raw[key])
        if not np.issubdtype(arr.dtype, np.number):
            continue
        arr = arr.astype(np.float32)
        arr = np.squeeze(arr)

        if arr.ndim != 2:
            continue
        if arr.shape[0] == 2 and arr.shape[1] >= 1:
            pts = arr.T
        elif arr.shape[1] == 2 and arr.shape[0] >= 1:
            pts = arr
        else:
            continue

        # Ignore tiny non-keypoint arrays such as bounding boxes only if a better
        # keypoint candidate exists. We keep all valid candidates and choose the
        # one with the largest number of points.
        candidates.append((key, pts))

    if not candidates:
        valid_keys = [k for k in raw.keys() if not k.startswith('__')]
        raise ValueError(f'No valid [K,2] or [2,K] keypoint array found in {mat_path}. Keys: {valid_keys}')

    key, pts = max(candidates, key=lambda x: x[1].shape[0])
    pts = pts.astype(np.float32)

    # PF-Willow MATLAB annotations are usually 1-based. Convert to 0-based when
    # coordinates look 1-based. If the file is already 0-based this does not fire.
    finite = pts[np.isfinite(pts)]
    if finite.size and finite.min() >= 1.0:
        pts = pts - 1.0

    return pts


def _pfw_collect_category_items(category):
    """Collect image paths and same-basename keypoints for one category."""
    cat_dir = os.path.join(LOCAL_PFW_DIR, category)
    img_paths = []
    for ext in ('*.png', '*.jpg', '*.jpeg', '*.bmp'):
        img_paths.extend(glob.glob(os.path.join(cat_dir, ext)))
    img_paths = sorted(img_paths)

    items = []
    skipped_no_mat = 0
    for img_path in img_paths:
        stem = os.path.splitext(img_path)[0]
        mat_path = stem + '.mat'
        if not os.path.exists(mat_path):
            skipped_no_mat += 1
            continue
        kps = _pfw_extract_keypoints_from_mat(mat_path)
        rel = os.path.relpath(img_path, LOCAL_PFW_DIR).replace('\\', '/')
        items.append((rel, kps))

    if skipped_no_mat:
        print(f'Warning: {category}: skipped {skipped_no_mat} images with no matching .mat file.')
    return items


def _pfw_load_pairs_and_kps(category):
    """
    Load PF-Willow pairs and corresponding keypoints for one category.

    Returns:
        pairs : list of (src_rel_path, trg_rel_path), relative to LOCAL_PFW_DIR
        XA    : np.ndarray [n_kps, n_pairs, 2], source keypoints, 0-based pixels
        XB    : np.ndarray [n_kps, n_pairs, 2], target keypoints, 0-based pixels
    """
    # ── Optional alternative layout: test_pairs_*.csv + annot/*.mat ─────────
    csv_path = os.path.join(LOCAL_PFW_DIR, f'test_pairs_{category}.csv')
    mat_path = os.path.join(LOCAL_PFW_DIR, 'annot', f'test_pairs_{category}.mat')
    if os.path.exists(csv_path) and os.path.exists(mat_path):
        df = pd.read_csv(csv_path, header=None, sep=None, engine='python', dtype=str)
        if df.shape[1] < 2:
            raise ValueError(f'Pair CSV has fewer than 2 columns: {csv_path}')
        pairs = [(str(row.iloc[0]).strip(), str(row.iloc[1]).strip())
                 for _, row in df.iterrows()]

        raw = scipy.io.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
        XA = np.asarray(raw['XA'], dtype=np.float32)
        XB = np.asarray(raw['XB'], dtype=np.float32)

        def _normalise(X):
            if X.ndim == 1:
                X = X.reshape(1, 1, 2)
            elif X.ndim == 2 and X.shape[0] == 2:
                X = X.T[:, np.newaxis, :]
            elif X.ndim == 3 and X.shape[0] == 2:
                X = X.transpose(1, 2, 0)
            return X - 1.0

        return pairs, _normalise(XA), _normalise(XB)

    # ── Screenshot/Drive layout: image + same-basename .mat per folder ──────
    items = _pfw_collect_category_items(category)
    if len(items) < 2:
        raise FileNotFoundError(
            f'Not enough annotated images in category {category}. Expected at least 2 in '
            f'{os.path.join(LOCAL_PFW_DIR, category)}'
        )

    pairs = []
    XA_list, XB_list = [], []

    # Deterministic same-subcategory adjacent pairs. This avoids exploding to all
    # pair combinations while still evaluating cross-image correspondence.
    for i in range(len(items) - 1):
        src_rel, src_kps = items[i]
        trg_rel, trg_kps = items[i + 1]
        n = min(src_kps.shape[0], trg_kps.shape[0])
        if n == 0:
            continue
        pairs.append((src_rel, trg_rel))
        XA_list.append(src_kps[:n])
        XB_list.append(trg_kps[:n])

    if not pairs:
        raise RuntimeError(f'No valid PF-Willow pairs generated for category {category}.')

    max_kps = max(x.shape[0] for x in XA_list)
    n_pairs = len(pairs)
    XA = np.full((max_kps, n_pairs, 2), np.nan, dtype=np.float32)
    XB = np.full((max_kps, n_pairs, 2), np.nan, dtype=np.float32)
    for j, (src_kps, trg_kps) in enumerate(zip(XA_list, XB_list)):
        n = min(src_kps.shape[0], trg_kps.shape[0], max_kps)
        XA[:n, j, :] = src_kps[:n]
        XB[:n, j, :] = trg_kps[:n]

    return pairs, XA, XB


def _pfw_find_image(rel_path):
    """Resolve a relative PF-Willow image path to an absolute local path."""
    rel_path = rel_path.replace('\\', '/')
    direct = os.path.join(LOCAL_PFW_DIR, rel_path)
    if os.path.exists(direct):
        return direct

    fname = os.path.basename(rel_path)
    hits  = glob.glob(os.path.join(LOCAL_PFW_DIR, '**', fname), recursive=True)
    return hits[0] if hits else None


def _pfw_feature_path(model_key, img_path):
    """Cache path for a PF-Willow frozen image feature."""
    rel = os.path.relpath(img_path, LOCAL_PFW_DIR)
    rel_no_ext = os.path.splitext(rel)[0]
    return os.path.join(LOCAL_PFW_FEATURE_ROOT, model_key, rel_no_ext + '.pt')


def _pfw_load_or_compute(model_key, img_path):
    """Load cached frozen PF-Willow feature or compute and cache it."""
    cache_path = _pfw_feature_path(model_key, img_path)
    if os.path.exists(cache_path):
        return torch.load(cache_path, map_location='cpu')

    feat = compute_feature(model_key, img_path)
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    torch.save(feat, cache_path)
    return feat


def evaluate_pfwillow(
    model_key,
    use_softargmax=False,
    softargmax_temp=0.01,
    softargmax_window=5,
    thresholds=None,
    bypass_cache=False,
    save_csv=True,
    label=None,
    max_pairs=None,
):
    """Evaluate semantic correspondence on PF-Willow."""
    if thresholds is None:
        thresholds = PCK_THRESHOLDS

    _device = 'cuda' if torch.cuda.is_available() else 'cpu'

    correct_kps    = {t: 0 for t in thresholds}
    total_kps      = 0
    per_image_rows = []
    skipped_missing_images = 0

    for category in tqdm(PFW_CATEGORIES, desc=f'PF-Willow eval [{model_key}]'):
        try:
            pairs, XA, XB = _pfw_load_pairs_and_kps(category)
        except (FileNotFoundError, ValueError, RuntimeError) as e:
            print(f"Warning: skipping category '{category}': {e}")
            continue

        n_pairs_cat = len(pairs)
        if max_pairs is not None:
            n_pairs_cat = min(n_pairs_cat, max_pairs)

        for pair_idx in range(n_pairs_cat):
            src_rel, trg_rel = pairs[pair_idx]

            src_path = _pfw_find_image(src_rel)
            trg_path = _pfw_find_image(trg_rel)

            if src_path is None or trg_path is None:
                skipped_missing_images += 1
                continue

            src_pil = Image.open(src_path).convert('RGB')
            trg_pil = Image.open(trg_path).convert('RGB')

            src_w, src_h = src_pil.size
            trg_w, trg_h = trg_pil.size
            norm_factor = max(trg_h, trg_w)

            if bypass_cache:
                raw_src = compute_feature(model_key, src_path)
                raw_trg = compute_feature(model_key, trg_path)
            else:
                raw_src = _pfw_load_or_compute(model_key, src_path)
                raw_trg = _pfw_load_or_compute(model_key, trg_path)

            f_src = F.normalize(raw_src.float(), dim=0).to(_device)
            f_trg = F.normalize(raw_trg.float(), dim=0).to(_device)

            _, src_fh, src_fw = f_src.shape
            _, trg_fh, trg_fw = f_trg.shape

            n_kps = XA.shape[0]
            image_correct = {t: 0 for t in thresholds}
            image_total   = 0

            for ki in range(n_kps):
                sx, sy = float(XA[ki, pair_idx, 0]), float(XA[ki, pair_idx, 1])
                tx, ty = float(XB[ki, pair_idx, 0]), float(XB[ki, pair_idx, 1])

                if any(math.isnan(v) for v in [sx, sy, tx, ty]):
                    continue
                if sx < 0 or sy < 0 or tx < 0 or ty < 0:
                    continue

                # Source pixel → source feature grid.
                feat_x = max(0, int(min(sx / src_w * src_fw, src_fw - 1)))
                feat_y = max(0, int(min(sy / src_h * src_fh, src_fh - 1)))

                target_feat = f_src[:, feat_y, feat_x]

                # Cosine similarity map over target features.
                sim = torch.einsum('c,chw->hw', target_feat, f_trg)

                if use_softargmax:
                    px_feat, py_feat = softargmax_2d(
                        sim,
                        temperature=softargmax_temp,
                        window_size=softargmax_window,
                    )
                    pred_x = ((px_feat.item() + 0.5) / trg_fw) * trg_w
                    pred_y = ((py_feat.item() + 0.5) / trg_fh) * trg_h
                else:
                    flat   = sim.argmax().item()
                    pred_x = ((flat % trg_fw  + 0.5) / trg_fw) * trg_w
                    pred_y = ((flat // trg_fw + 0.5) / trg_fh) * trg_h

                dist = math.sqrt((pred_x - tx) ** 2 + (pred_y - ty) ** 2)

                total_kps   += 1
                image_total += 1

                for t in thresholds:
                    if dist <= t * norm_factor:
                        correct_kps[t]   += 1
                        image_correct[t] += 1

            per_image_rows.append({
                'category':  category,
                'src_image': src_rel,
                'trg_image': trg_rel,
                'total_kps': image_total,
                **{
                    f'PCK@{t}': (
                        image_correct[t] / image_total * 100.0
                        if image_total > 0 else 0.0
                    )
                    for t in thresholds
                },
            })

    per_keypoint = {
        f'PCK@{t}': (
            correct_kps[t] / total_kps * 100.0
            if total_kps > 0 else 0.0
        )
        for t in thresholds
    }

    per_category_mean = {}
    if per_image_rows:
        per_img_df_tmp = pd.DataFrame(per_image_rows)
        for t in thresholds:
            per_category_mean[f'macro_category_PCK@{t}'] = (
                per_img_df_tmp.groupby('category')[f'PCK@{t}'].mean().mean()
            )

    print()
    print('PF-Willow evaluation finished')
    print('------------------------------')
    print('Model                  :', model_key)
    print('Use softargmax         :', use_softargmax)
    print('Total evaluated pairs  :', len(per_image_rows))
    print('Total evaluated kps    :', total_kps)
    print('Skipped missing images :', skipped_missing_images)

    print()
    print('PF-Willow per-keypoint PCK:')
    for t in thresholds:
        print(f"  PCK@{t}: {per_keypoint[f'PCK@{t}']:.2f}%")

    if per_category_mean:
        print('PF-Willow macro-category PCK:')
        for t in thresholds:
            print(f"  macro PCK@{t}: {per_category_mean[f'macro_category_PCK@{t}']:.2f}%")

    if save_csv:
        tag  = label or ('softargmax' if use_softargmax else 'argmax')
        csv1 = os.path.join(RESULTS_ROOT, f'pfwillow_{model_key}_{tag}_per_image.csv')
        csv2 = os.path.join(RESULTS_ROOT, f'pfwillow_{model_key}_{tag}_per_keypoint.csv')

        pd.DataFrame(per_image_rows).to_csv(csv1, index=False)
        pd.DataFrame([{
            'model_key': model_key,
            'dataset':   'PF-Willow',
            'total_keypoints':   total_kps,
            'total_image_pairs': len(per_image_rows),
            **per_keypoint,
            **per_category_mean,
        }]).to_csv(csv2, index=False)

        print()
        print('Saved:', csv1)
        print('Saved:', csv2)

    return {
        'model_key':   model_key,
        'dataset':     'PF-Willow',
        'use_softargmax': use_softargmax,
        'per_keypoint':   per_keypoint,
        'per_category_mean': per_category_mean,
        'per_image':      per_image_rows,
        'total_keypoints': total_kps,
        'total_image_pairs': len(per_image_rows),
        'skipped_missing_images': skipped_missing_images,
    }


print('PF-Willow data loader and evaluate_pfwillow() defined.')


PF-Willow data loader and evaluate_pfwillow() defined.


In [33]:
# Stage 4 — PF-Willow smoke test.
# Runs a small DINOv2 frozen evaluation to validate paths, keypoint parsing, pair generation, and feature extraction before the full run.

reload_pretrained("dinov2_vitl14")
_get_backbone("dinov2_vitl14").to(device).eval()

pfw_test_res = evaluate_pfwillow(
    "dinov2_vitl14",
    use_softargmax=False,
    bypass_cache=True,   # force fresh computation for the smoke test
    save_csv=False,
    max_pairs=2,         # only 2 pairs per category — fast sanity check
)

pfw_smoke_pairs = int(pfw_test_res.get("total_image_pairs", 0))
pfw_smoke_kps   = int(pfw_test_res.get("total_keypoints",   0))

print("\nPF-Willow smoke-test summary")
print("-" * 40)
print(f"Evaluated image pairs : {pfw_smoke_pairs}")
print(f"Evaluated keypoints   : {pfw_smoke_kps}")
print("Per-keypoint PCK      :", pfw_test_res.get("per_keypoint", {}))

assert pfw_smoke_pairs > 0, (
    "PF-Willow smoke test failed: no image pairs were evaluated. "
    "Check LOCAL_PFW_DIR, CSV files, and image paths."
)
assert pfw_smoke_kps > 0, (
    "PF-Willow smoke test failed: no valid keypoints were evaluated. "
    "Check .mat keypoint parsing (XA/XB shape and 0-basing)."
)

PFW_SMOKE_TEST_PASSED = True
print("\nPF-Willow smoke test passed.")

  Reloaded pretrained weights: dinov2_vitl14


PF-Willow eval [dinov2_vitl14]:   0%|          | 0/10 [00:00<?, ?it/s]


PF-Willow evaluation finished
------------------------------
Model                  : dinov2_vitl14
Use softargmax         : False
Total evaluated pairs  : 20
Total evaluated kps    : 200
Skipped missing images : 0

PF-Willow per-keypoint PCK:
  PCK@0.05: 55.50%
  PCK@0.1: 81.50%
  PCK@0.15: 89.00%
  PCK@0.2: 92.50%
PF-Willow macro-category PCK:
  macro PCK@0.05: 55.50%
  macro PCK@0.1: 81.50%
  macro PCK@0.15: 89.00%
  macro PCK@0.2: 92.50%

PF-Willow smoke-test summary
----------------------------------------
Evaluated image pairs : 20
Evaluated keypoints   : 200
Per-keypoint PCK      : {'PCK@0.05': 55.50000000000001, 'PCK@0.1': 81.5, 'PCK@0.15': 89.0, 'PCK@0.2': 92.5}

PF-Willow smoke test passed.


In [34]:
# Stage 4 — full PF-Willow evaluation.
# Compares frozen, fine-tuned argmax, and fine-tuned soft-argmax conditions for every backbone.

required_stage_vars = ["BEST_CKPTS", "S3_BEST_CFG", "MODEL_SPECS", "PCK_THRESHOLDS"]
missing_stage_vars  = [v for v in required_stage_vars if v not in globals()]
if missing_stage_vars:
    raise RuntimeError(
        "Missing required variables from earlier stages: " + ", ".join(missing_stage_vars) +
        ". Re-run the setup and Stages 1–3 before running the PF-Willow extension."
    )

if not globals().get("PFW_SMOKE_TEST_PASSED", False):
    raise RuntimeError(
        "Run the PF-Willow smoke test cell first. "
        "The smoke test must pass before launching the full evaluation."
    )

PFW_RESULTS = []   # accumulates one dict per (model, condition)

for model_key in MODEL_SPECS:
    print(f"\n{'='*65}")
    print(f"  PF-Willow evaluation: {model_key}")
    print(f"{'='*65}")

    # ── (A) FROZEN ───────────────────────────────────────────────────────────
    print("\n[A] Frozen baseline (argmax)")
    reload_pretrained(model_key)
    _get_backbone(model_key).to(device).eval()

    res_frozen = evaluate_pfwillow(
        model_key,
        use_softargmax=False,
        bypass_cache=False,   # use feature cache for speed
        save_csv=True,
        label='frozen_argmax',
    )
    PFW_RESULTS.append({
        'model_key':   model_key,
        'condition':   'frozen_argmax',
        'N_finetuned': 0,
        **res_frozen['per_keypoint'],
        **res_frozen.get('per_category_mean', {}),
    })

    # ── (B) FINE-TUNED + ARGMAX ───────────────────────────────────────────────
    info = BEST_CKPTS.get(model_key, {})
    N    = info.get('N', 0)
    ckpt = info.get('ckpt_path')

    print(f"\n[B] Fine-tuned N={N} + argmax")
    reload_pretrained(model_key)
    if N > 0 and ckpt and os.path.exists(ckpt):
        backbone = _get_backbone(model_key)
        backbone.load_state_dict(torch.load(ckpt, map_location='cpu'))
        backbone.to(device).eval()
        print(f"  Loaded: {ckpt}")
    else:
        print(f"  No checkpoint found for N={N} — using pretrained weights.")

    res_ft = evaluate_pfwillow(
        model_key,
        use_softargmax=False,
        bypass_cache=True,    # fine-tuned weights differ from cached frozen features
        save_csv=True,
        label=f'ft_N{N}_argmax',
    )
    PFW_RESULTS.append({
        'model_key':   model_key,
        'condition':   f'ft_N{N}_argmax',
        'N_finetuned': N,
        **res_ft['per_keypoint'],
        **res_ft.get('per_category_mean', {}),
    })

    # ── (C) FINE-TUNED + SOFT-ARGMAX ─────────────────────────────────────────
    cfg       = S3_BEST_CFG.get(model_key, {'temperature': 0.01, 'window_size': 5})
    best_temp = cfg['temperature']
    best_win  = int(cfg['window_size'])
    print(f"\n[C] Fine-tuned N={N} + soft-argmax (temp={best_temp}, win={best_win})")

    res_sa = evaluate_pfwillow(
        model_key,
        use_softargmax=True,
        softargmax_temp=best_temp,
        softargmax_window=best_win,
        bypass_cache=True,
        save_csv=True,
        label=f'ft_N{N}_softargmax',
    )
    PFW_RESULTS.append({
        'model_key':    model_key,
        'condition':    f'ft_N{N}_softargmax',
        'N_finetuned':  N,
        'temperature':  best_temp,
        'window_size':  best_win,
        **res_sa['per_keypoint'],
        **res_sa.get('per_category_mean', {}),
    })

# ── Save combined summary ─────────────────────────────────────────────────────
pfw_summary_path = os.path.join(RESULTS_ROOT, 'pfwillow_extension_summary.csv')
pfw_df = pd.DataFrame(PFW_RESULTS)
pfw_df.to_csv(pfw_summary_path, index=False)
print(f"\nSaved PF-Willow summary: {pfw_summary_path}")

# ── Sync PF-Willow feature cache to Drive ────────────────────────────────────
print("\nSyncing PF-Willow feature cache to Drive …")
os.makedirs(DRIVE_PFW_FEATURE_ROOT, exist_ok=True)
for root, dirs, files in os.walk(LOCAL_PFW_FEATURE_ROOT):
    rel      = os.path.relpath(root, LOCAL_PFW_FEATURE_ROOT)
    dst_root = os.path.join(DRIVE_PFW_FEATURE_ROOT, rel)
    os.makedirs(dst_root, exist_ok=True)
    for fn in files:
        src_f = os.path.join(root, fn)
        dst_f = os.path.join(dst_root, fn)
        if not os.path.exists(dst_f):
            shutil.copy2(src_f, dst_f)
print("Feature cache synced.")


  PF-Willow evaluation: dinov2_vitl14

[A] Frozen baseline (argmax)
  Reloaded pretrained weights: dinov2_vitl14


PF-Willow eval [dinov2_vitl14]:   0%|          | 0/10 [00:00<?, ?it/s]


PF-Willow evaluation finished
------------------------------
Model                  : dinov2_vitl14
Use softargmax         : False
Total evaluated pairs  : 90
Total evaluated kps    : 900
Skipped missing images : 0

PF-Willow per-keypoint PCK:
  PCK@0.05: 52.56%
  PCK@0.1: 77.33%
  PCK@0.15: 86.78%
  PCK@0.2: 91.22%
PF-Willow macro-category PCK:
  macro PCK@0.05: 52.56%
  macro PCK@0.1: 77.33%
  macro PCK@0.15: 86.78%
  macro PCK@0.2: 91.22%

Saved: /content/drive/MyDrive/results/pfwillow_dinov2_vitl14_frozen_argmax_per_image.csv
Saved: /content/drive/MyDrive/results/pfwillow_dinov2_vitl14_frozen_argmax_per_keypoint.csv

[B] Fine-tuned N=4 + argmax
  Reloaded pretrained weights: dinov2_vitl14
  Loaded: /content/drive/MyDrive/checkpoints/finetuned/dinov2_vitl14_last4.pth


PF-Willow eval [dinov2_vitl14]:   0%|          | 0/10 [00:00<?, ?it/s]


PF-Willow evaluation finished
------------------------------
Model                  : dinov2_vitl14
Use softargmax         : False
Total evaluated pairs  : 90
Total evaluated kps    : 900
Skipped missing images : 0

PF-Willow per-keypoint PCK:
  PCK@0.05: 61.89%
  PCK@0.1: 84.22%
  PCK@0.15: 91.44%
  PCK@0.2: 94.22%
PF-Willow macro-category PCK:
  macro PCK@0.05: 61.89%
  macro PCK@0.1: 84.22%
  macro PCK@0.15: 91.44%
  macro PCK@0.2: 94.22%

Saved: /content/drive/MyDrive/results/pfwillow_dinov2_vitl14_ft_N4_argmax_per_image.csv
Saved: /content/drive/MyDrive/results/pfwillow_dinov2_vitl14_ft_N4_argmax_per_keypoint.csv

[C] Fine-tuned N=4 + soft-argmax (temp=0.05, win=7)


...


PF-Willow evaluation finished
------------------------------
Model                  : sam_vit_b_res1024
Use softargmax         : False
Total evaluated pairs  : 90
Total evaluated kps    : 900
Skipped missing images : 0

PF-Willow per-keypoint PCK:
  PCK@0.05: 30.11%
  PCK@0.1: 46.11%
  PCK@0.15: 54.78%
  PCK@0.2: 63.56%
PF-Willow macro-category PCK:
  macro PCK@0.05: 30.11%
  macro PCK@0.1: 46.11%
  macro PCK@0.15: 54.78%
  macro PCK@0.2: 63.56%

Saved: /content/drive/MyDrive/results/pfwillow_sam_vit_b_res1024_frozen_argmax_per_image.csv
Saved: /content/drive/MyDrive/results/pfwillow_sam_vit_b_res1024_frozen_argmax_per_keypoint.csv

[B] Fine-tuned N=2 + argmax
  Reloaded pretrained weights: sam_vit_b_res1024
  Loaded: /content/drive/MyDrive/checkpoints/finetuned/sam_vit_b_res1024_last2.pth


PF-Willow eval [sam_vit_b_res1024]:   0%|          | 0/10 [00:00<?, ?it/s]


PF-Willow evaluation finished
------------------------------
Model                  : sam_vit_b_res1024
Use softargmax         : False
Total evaluated pairs  : 90
Total evaluated kps    : 900
Skipped missing images : 0

PF-Willow per-keypoint PCK:
  PCK@0.05: 37.56%
  PCK@0.1: 55.00%
  PCK@0.15: 65.11%
  PCK@0.2: 72.56%
PF-Willow macro-category PCK:
  macro PCK@0.05: 37.56%
  macro PCK@0.1: 55.00%
  macro PCK@0.15: 65.11%
  macro PCK@0.2: 72.56%

Saved: /content/drive/MyDrive/results/pfwillow_sam_vit_b_res1024_ft_N2_argmax_per_image.csv
Saved: /content/drive/MyDrive/results/pfwillow_sam_vit_b_res1024_ft_N2_argmax_per_keypoint.csv

[C] Fine-tuned N=2 + soft-argmax (temp=0.05, win=11)


PF-Willow eval [sam_vit_b_res1024]:   0%|          | 0/10 [00:00<?, ?it/s]


PF-Willow evaluation finished
------------------------------
Model                  : sam_vit_b_res1024
Use softargmax         : True
Total evaluated pairs  : 90
Total evaluated kps    : 900
Skipped missing images : 0

PF-Willow per-keypoint PCK:
  PCK@0.05: 38.33%
  PCK@0.1: 55.78%
  PCK@0.15: 65.67%
  PCK@0.2: 73.00%
PF-Willow macro-category PCK:
  macro PCK@0.05: 38.33%
  macro PCK@0.1: 55.78%
  macro PCK@0.15: 65.67%
  macro PCK@0.2: 73.00%

Saved: /content/drive/MyDrive/results/pfwillow_sam_vit_b_res1024_ft_N2_softargmax_per_image.csv
Saved: /content/drive/MyDrive/results/pfwillow_sam_vit_b_res1024_ft_N2_softargmax_per_keypoint.csv

Saved PF-Willow summary: /content/drive/MyDrive/results/pfwillow_extension_summary.csv

Syncing PF-Willow feature cache to Drive …
Feature cache synced.


In [35]:
# Stage 4 — display PF-Willow results.
# Prints the extension summary table and a per-category PCK@0.1 breakdown for the selected fine-tuned condition.

pfw_df = pd.read_csv(os.path.join(RESULTS_ROOT, 'pfwillow_extension_summary.csv'))

print()
print("=" * 95)
print("STEP 4 RESULTS — PF-Willow Semantic Correspondence (PCK@α · max(H,W))")
print("=" * 95)

COND_LABELS_PFW = {'frozen_argmax': 'Frozen       (argmax)'}
for mk in pfw_df['model_key'].unique():
    for row in pfw_df[pfw_df['model_key'] == mk].itertuples():
        N = getattr(row, 'N_finetuned', 0)
        if 'ft' in row.condition and 'softargmax' not in row.condition:
            COND_LABELS_PFW[row.condition] = f'FT N={N}     (argmax)'
        elif 'ft' in row.condition and 'softargmax' in row.condition:
            COND_LABELS_PFW[row.condition] = f'FT N={N}     (soft-argmax)'

header = f"{'Model + Condition':<46} " + "   ".join(f"PCK@{t}" for t in PCK_THRESHOLDS)
print(header)
print("-" * 95)

for mk in pfw_df['model_key'].unique():
    sub = pfw_df[pfw_df['model_key'] == mk]
    for _, row in sub.iterrows():
        cond_label = COND_LABELS_PFW.get(row['condition'], row['condition'])
        label      = f"{mk}  [{cond_label}]"
        pck_vals   = "   ".join(f"{row[f'PCK@{t}']:>8.2f}%" for t in PCK_THRESHOLDS)
        print(f"{label:<64} {pck_vals}")
    print()

print("=" * 95)

# ── Cross-dataset commentary ─────────────────────────────────────────────────
print()
print("Cross-dataset generalisation (SPair-71k → PF-Willow):")
print("  • Positive Δ (fine-tuned vs frozen) means SPair-71k fine-tuning")
print("    generalises to PF-Willow without any PF-Willow training.")
print("  • DINOv2/v3 typically retain strong generalisation; SAM may lag.")
print("  • Soft-argmax gains at PCK@0.05 should mirror the SPair-71k pattern.")

# ── Per-category breakdown ────────────────────────────────────────────────────
print()
print("Per-category PCK@0.1 (fine-tuned best condition per backbone):")
print("-" * 80)

for model_key in pfw_df['model_key'].unique():
    sub     = pfw_df[pfw_df['model_key'] == model_key]
    ft_rows = sub[sub['condition'].str.contains('softargmax')]
    if ft_rows.empty:
        ft_rows = sub[sub['condition'].str.contains('ft')]
    if ft_rows.empty:
        continue

    chosen_cond = ft_rows.iloc[0]['condition']
    per_img_csv = os.path.join(
        RESULTS_ROOT, f'pfwillow_{model_key}_{chosen_cond}_per_image.csv'
    )
    if not os.path.exists(per_img_csv):
        continue

    per_img_df = pd.read_csv(per_img_csv)
    cat_pck = (
        per_img_df.groupby('category')['PCK@0.1']
        .mean()
        .sort_values(ascending=False)
    )
    print(f"\n{model_key} [{chosen_cond}]:")
    for cat, val in cat_pck.items():
        bar = '#' * int(val / 2)
        print(f"  {cat:<20} {val:>6.2f}%  {bar}")

print()
print("All PF-Willow results saved. PF-Willow feature cache was synced in the previous cell.")


STEP 4 RESULTS — PF-Willow Semantic Correspondence (PCK@α · max(H,W))
Model + Condition                              PCK@0.05   PCK@0.1   PCK@0.15   PCK@0.2
-----------------------------------------------------------------------------------------------
dinov2_vitl14  [Frozen       (argmax)]                              52.56%      77.33%      86.78%      91.22%
dinov2_vitl14  [FT N=4     (argmax)]                                61.89%      84.22%      91.44%      94.22%
dinov2_vitl14  [FT N=4     (soft-argmax)]                           62.22%      85.56%      91.67%      94.33%

dinov3_vitb16  [Frozen       (argmax)]                              62.89%      85.67%      93.56%      96.22%
dinov3_vitb16  [FT N=2     (argmax)]                                66.67%      87.78%      94.11%      96.67%
dinov3_vitb16  [FT N=2     (soft-argmax)]                           69.00%      89.22%      94.11%      96.67%

sam_vit_b_res1024  [Frozen       (argmax)]                          30.11%    